# 🧬 DTIAM Modern Stack V5-1.3
**Drug-Target Interaction / Affinity Prediction Pipeline**

### Cambios sobre V5-1.2:
- ✅ **UnifiedRegistry** — diccionario canónico cross-dataset, hash SHA-256 por SMILES/secuencia
- ✅ **Serialización segura** — reemplaza PKL/dill por HDF5 (embeddings) + JSON (metadatos)
- ✅ **DataStandardizer v2** — integrado con el registro, limpia prefijos de todos los datasets
- ✅ **load_fold_data fix** — robusto a IDs sin features (bug hetionet corregido)
- ✅ **ESM-2 via HuggingFace** — elimina `import esm` (fair-esm deprecado)
- ✅ **kfold_validation mejorado** — guarda bundle JSON al terminar, benchmarking automático
- ✅ **DTIPredictor v2** — carga desde HDF5, sin pickle en ningún punto del pipeline
- ✅ **predict(smiles, sequence)** — función lista para conectar a GUI

### Arquitectura:
```
SMILES  → canonicalize → UnifiedRegistry → ChemBERTa → embedding → HDF5
Seq AA  → clean        → UnifiedRegistry → ESM-2      → embedding → HDF5
                                                              ↓
                                                   AutoGluon Ensemble
                                                              ↓
                                                   score DTI / afinidad
```

---
## 📦 CELDA 0: Dependencias

In [1]:
# pip install safetensors h5py transformers selfies autogluon gradio rdkit biopython
# (ver dtiam_mod_stack_v5.yaml para entorno completo)
import subprocess, sys

REQUIRED = ["safetensors", "h5py"]
for pkg in REQUIRED:
    try:
        __import__(pkg)
        print(f"✅ {pkg}")
    except ImportError:
        print(f"📦 Instalando {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"✅ {pkg} instalado")


✅ safetensors
✅ h5py


---
## ⚙️ CELDA 1: Configuración Global

In [1]:
import os, sys, json, time, re, hashlib, random, itertools, pickle, warnings
import numpy as np
import pandas as pd
import torch
import h5py
import selfies as sf
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from tqdm import tqdm
from math import ceil
from collections import OrderedDict
from pathlib import Path
from safetensors.numpy import save_file as st_save, load_file as st_load

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    print(f"🖥️  GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
    if vram_gb >= 10:
        recommended_esm, recommended_chem = "large", "chemberta_medium"
        print("🏆 Config RECOMENDADA: ESM-2 650M + ChemBERTa-77M")
    elif vram_gb >= 6:
        recommended_esm, recommended_chem = "medium", "chemberta_medium"
        print("✅ Config MEDIA: ESM-2 150M + ChemBERTa-77M")
    else:
        recommended_esm, recommended_chem = "light", "chemberta_light"
        print("🔋 Config LIGERA: ESM-2 8M + ChemBERTa-10M")
else:
    print("⚠️  Sin GPU — CPU mode")
    vram_gb  = 0
    recommended_esm, recommended_chem = "light", "chemberta_light"

print(f"📍 Device: {DEVICE}")

# ── Rutas ──────────────────────────────────────────────────────────────────
# ── Detección automática de rutas ─────────────────────────────────────────────
# Estrategia en 3 niveles (sin hardcodear nada):
#
#  Nivel 1 — Autodetección: busca la carpeta raíz del proyecto subiendo desde
#            este notebook hasta encontrar la estructura esperada (code/ + data/)
#
#  Nivel 2 — Variable de entorno: si defines DTIAM_ROOT en tu shell antes de
#            abrir Jupyter, se usa esa ruta directamente:
#              export DTIAM_ROOT=/ruta/a/DTIAM-ModStack
#
#  Nivel 3 — Diálogo interactivo: si los dos niveles anteriores fallan, abre
#            un selector de carpeta (GUI) o pide la ruta por input().

def _find_project_root(start: Path, marker_dirs=("code", "data", "results", "models")) -> Path | None:
    """
    Sube desde `start` hasta encontrar un directorio que contenga
    todas las carpetas marcadoras del proyecto.
    Evita subir más allá de la raíz del sistema de archivos.
    """
    current = start.resolve()
    for _ in range(10):  # máximo 10 niveles hacia arriba
        if all((current / m).exists() for m in marker_dirs):
            return current
        parent = current.parent
        if parent == current:   # llegamos a la raíz del sistema
            break
        current = parent
    return None


def _ask_root_gui() -> Path | None:
    """Intenta abrir un diálogo gráfico de selección de carpeta."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root_tk = tk.Tk()
        root_tk.withdraw()          # ocultar ventana principal
        root_tk.attributes("-topmost", True)
        selected = filedialog.askdirectory(
            title="Selecciona la carpeta raíz del proyecto (DTIAM-ModStack/)"
        )
        root_tk.destroy()
        return Path(selected) if selected else None
    except Exception:
        return None


def resolve_project_root() -> Path:
    """
    Devuelve la ruta raíz del proyecto (la carpeta que contiene code/, data/, etc.)
    Intenta los 3 niveles en orden y lanza un error claro si ninguno funciona.
    """
    # Nivel 1 — Autodetección desde el notebook
    notebook_dir = Path(globals().get("__vsc_ipynb_file__", "")).parent
    if not notebook_dir.exists():
        # En Jupyter clásico / JupyterLab
        try:
            import IPython
            notebook_dir = Path(IPython.get_ipython().starting_dir)
        except Exception:
            notebook_dir = Path.cwd()

    auto = _find_project_root(notebook_dir)
    if auto:
        print(f"✅ Proyecto detectado automáticamente: {auto}")
        return auto

    # Nivel 2 — Variable de entorno DTIAM_ROOT
    env_root = os.environ.get("DTIAM_ROOT", "")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if p.exists():
            print(f"✅ Usando DTIAM_ROOT del entorno: {p}")
            return p
        else:
            print(f"⚠️  DTIAM_ROOT={env_root} no existe, ignorando.")

    # Nivel 3a — Diálogo gráfico (tkinter)
    print("⚠️  No se detectó la raíz automáticamente. Abriendo selector de carpeta...")
    gui_path = _ask_root_gui()
    if gui_path and gui_path.exists():
        print(f"✅ Raíz seleccionada: {gui_path}")
        return gui_path

    # Nivel 3b — Input de texto como último recurso
    print("⚠️  Sin entorno gráfico. Ingresa la ruta manualmente.")
    manual = input("📁 Ruta a la carpeta raíz del proyecto (DTIAM-ModStack/): ").strip()
    p = Path(manual).expanduser().resolve()
    if p.exists():
        return p

    raise FileNotFoundError(
        "❌ No se pudo determinar la raíz del proyecto.\n"
        "   Opciones:\n"
        "   1. Abre Jupyter desde dentro de DTIAM-ModStack/code/\n"
        "   2. Define: export DTIAM_ROOT=/ruta/a/DTIAM-ModStack\n"
        "   3. Ingresa la ruta manualmente cuando se pida."
    )


# ── Resolver y configurar rutas ───────────────────────────────────────────────
PROJECT_ROOT  = resolve_project_root()
CODE_PATH     = PROJECT_ROOT / "code"
MASTER_PATH   = PROJECT_ROOT / "data"  / "master_features"
REGISTRY_PATH = PROJECT_ROOT / "data"  / "registry"
RESULTS_PATH  = PROJECT_ROOT / "results"
MODELS_PATH   = PROJECT_ROOT / "models" / "trained"

# Cambiar el directorio de trabajo a code/ (donde vive el notebook)
os.chdir(CODE_PATH)
print(f"📁 Working dir: {Path.cwd()}")

# Crear carpetas necesarias
for p in [MASTER_PATH, REGISTRY_PATH, RESULTS_PATH, MODELS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Estructura del proyecto:")
print(f"   ROOT    : {PROJECT_ROOT}")
print(f"   code/   : {CODE_PATH}")
print(f"   data/   : {PROJECT_ROOT / 'data'}")
print(f"   results/: {RESULTS_PATH}")
print(f"   models/ : {MODELS_PATH}")

# ── PLM Catalog ────────────────────────────────────────────────────────────
PLM_CATALOG = {
    "compound": {
        "chemberta_light":  {"name": "DeepChem/ChemBERTa-10M-MTR",        "params": "10M",  "vram_req": "<2GB",  "embed_dim": 384,  "input_format": "smiles"},
        "chemberta_medium": {"name": "DeepChem/ChemBERTa-77M-MTR",        "params": "77M",  "vram_req": "~3GB",  "embed_dim": 768,  "input_format": "smiles"},
        "chemberta_base":   {"name": "seyonec/ChemBERTa-zinc-base-v1",    "params": "86M",  "vram_req": "~3GB",  "embed_dim": 768,  "input_format": "smiles"},
        "selformer":        {"name": "HUBioinfo/SELFormer",                "params": "86M",  "vram_req": "~3GB",  "embed_dim": 768,  "input_format": "selfies"},
    },
    "protein": {
        "light":  {"name": "facebook/esm2_t6_8M_UR50D",    "params": "8M",   "vram_req": "<2GB",  "embed_dim": 320},
        "medium": {"name": "facebook/esm2_t30_150M_UR50D", "params": "150M", "vram_req": "~4GB",  "embed_dim": 640},
        "large":  {"name": "facebook/esm2_t33_650M_UR50D", "params": "650M", "vram_req": "~8GB",  "embed_dim": 1280},
    }
}
# Aliases cortos
for k in ["light", "medium", "base"]:
    PLM_CATALOG["compound"][k] = PLM_CATALOG["compound"].get(f"chemberta_{k}",
                                  PLM_CATALOG["compound"].get("chemberta_medium"))

print("\n📋 PLM_CATALOG listo.")
print("   Compuestos:", list(PLM_CATALOG["compound"].keys()))
print("   Proteínas: ", list(PLM_CATALOG["protein"].keys()))


🖥️  GPU: NVIDIA GeForce RTX 2070 SUPER | VRAM: 8.2 GB
✅ Config MEDIA: ESM-2 150M + ChemBERTa-77M
📍 Device: cuda
✅ Proyecto detectado automáticamente: /home/yosh/Documentos/TDA-Net
📁 Working dir: /home/yosh/Documentos/TDA-Net/code

📂 Estructura del proyecto:
   ROOT    : /home/yosh/Documentos/TDA-Net
   code/   : /home/yosh/Documentos/TDA-Net/code
   data/   : /home/yosh/Documentos/TDA-Net/data
   results/: /home/yosh/Documentos/TDA-Net/results
   models/ : /home/yosh/Documentos/TDA-Net/models/trained

📋 PLM_CATALOG listo.
   Compuestos: ['chemberta_light', 'chemberta_medium', 'chemberta_base', 'selformer', 'light', 'medium', 'base']
   Proteínas:  ['light', 'medium', 'large']


---
## 🗂️ CELDA 2: UnifiedRegistry — Diccionario Canónico Cross-Dataset

In [21]:
class UnifiedRegistry:
    """
    Registro canónico de compuestos y proteínas cross-dataset.

    Problema que resuelve:
        El mismo fármaco puede aparecer en Yamanishi como 'D00448',
        en HetioNet como 'Compound::DB00328', y en Davis con otro ID.
        Si sus SMILES canónicos coinciden, son la MISMA entidad y deben
        compartir el mismo embedding.

    Solución:
        - Compuestos: clave = SHA-256(SMILES canónico RDKit)
        - Proteínas:  clave = SHA-256(secuencia AA limpia)
        - Guarda mapping.json con tabla de equivalencias ID_original → ID_interno
        - Sin pickle en ningún punto — solo JSON y HDF5

    Uso:
        registry = UnifiedRegistry()
        registry.load()                     # cargar si ya existe
        uid = registry.get_compound_uid(smiles, original_id="D00448", source="yamanishi")
        uid = registry.get_protein_uid(seq,    original_id="hsa:1234",  source="hetionet")
        registry.save()
    """

    def __init__(self, path: Path = REGISTRY_PATH):
        self.path = Path(path)
        self.path.mkdir(parents=True, exist_ok=True)

        # hash → UID interno  (e.g. "COMP_000042")
        self.comp_hash_to_uid:  dict[str, str] = {}
        self.prot_hash_to_uid:  dict[str, str] = {}

        # UID → hash canónico (inverso)
        self.comp_uid_to_hash:  dict[str, str] = {}
        self.prot_uid_to_hash:  dict[str, str] = {}

        # UID → {source_id: dataset, ...}  (trazabilidad cross-dataset)
        self.comp_aliases:      dict[str, dict] = {}
        self.prot_aliases:      dict[str, dict] = {}

        # UID → SMILES canónico / secuencia limpia (para re-generar embeddings)
        self.comp_canonical:    dict[str, str]  = {}
        self.prot_canonical:    dict[str, str]  = {}

    # ── Hashing ──────────────────────────────────────────────────────────────
    @staticmethod
    def _hash(text: str) -> str:
        return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]  # 16 hex = 64 bits, suficiente

    # ── SMILES helpers ───────────────────────────────────────────────────────
    @staticmethod
    def canonicalize_smiles(smi: str) -> str | None:
        """
        Canonicaliza un SMILES con 3 niveles de tolerancia:

        Intento 1 — Estricto (99% de los casos):
            Chem.MolFromSmiles() en modo normal. Falla con hidrógenos
            explícitos en posiciones inusuales (ej. [H][C@]12...).

        Intento 2 — Tolerante (SMILES con H explícitos):
            sanitize=False + RemoveHs() + SanitizeMol() manual.
            Resuelve los warnings "not removing hydrogen atom without
            neighbors" que aparecen en HetioNet y algunos datasets
            de ChEMBL.

        Intento 3 — Permisivo (casos extremos):
            Sanitización parcial que omite validación de propiedades.
            Último recurso antes de descartar el SMILES.

        Returns:
            SMILES canónico (str) o None si genuinamente inválido.
        """
        if not smi or not isinstance(smi, str):
            return None
        smi = smi.strip()

        # ── Intento 1: modo estricto ──────────────────────────────────────
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                return Chem.MolToSmiles(mol)
        except Exception:
            pass

        # ── Intento 2: tolerante — remueve H explícitos problemáticos ────
        # Resuelve casos como [H][C@]12... que son válidos pero RDKit
        # no acepta en modo estricto por neighbors incompletos
        try:
            mol = Chem.MolFromSmiles(smi, sanitize=False)
            if mol is not None:
                mol = Chem.RemoveHs(mol, sanitize=False)
                Chem.SanitizeMol(mol)
                return Chem.MolToSmiles(mol)
        except Exception:
            pass

        # ── Intento 3: permisivo — sanitización parcial ───────────────────
        # Valida valencia y aromaticidad pero omite propiedades calculadas
        try:
            mol = Chem.MolFromSmiles(smi, sanitize=False)
            if mol is not None:
                Chem.SanitizeMol(
                    mol,
                    Chem.SanitizeFlags.SANITIZE_ALL ^
                    Chem.SanitizeFlags.SANITIZE_PROPERTIES
                )
                return Chem.MolToSmiles(mol)
        except Exception:
            pass

        return None   # genuinamente inválido

    @staticmethod
    def smiles_to_selfies(smi: str) -> str | None:
        canon = UnifiedRegistry.canonicalize_smiles(smi)
        if canon is None:
            return None
        try:
            return sf.encoder(canon)
        except Exception:
            return None

    # ── Similaridad Tanimoto (compuestos) ────────────────────────────────────
    @staticmethod
    def tanimoto(smi1: str, smi2: str, radius: int = 2) -> float:
        """Similaridad Tanimoto entre dos SMILES usando Morgan fingerprints."""
        mol1 = Chem.MolFromSmiles(smi1)
        mol2 = Chem.MolFromSmiles(smi2)
        if mol1 is None or mol2 is None:
            return 0.0
        fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, radius, nBits=2048)
        fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, radius, nBits=2048)
        return DataStructs.TanimotoSimilarity(fp1, fp2)

    # ── Similaridad de secuencia (proteínas) ─────────────────────────────────
    @staticmethod
    def sequence_identity(seq1: str, seq2: str) -> float:
        """Identidad de secuencia simple (posición por posición)."""
        if not seq1 or not seq2:
            return 0.0
        min_len = min(len(seq1), len(seq2))
        matches = sum(a == b for a, b in zip(seq1[:min_len], seq2[:min_len]))
        return matches / max(len(seq1), len(seq2))

    # ── Registro de compuestos ────────────────────────────────────────────────
    def get_compound_uid(
        self,
        smiles: str,
        original_id: str = "",
        source: str = ""
    ) -> str | None:
        """
        Retorna el UID interno del compuesto.
        Si no existe, lo crea. Si el SMILES es inválido, retorna None.
        """
        canon = self.canonicalize_smiles(smiles)
        if canon is None:
            return None

        h = self._hash(canon)
        if h not in self.comp_hash_to_uid:
            uid = f"COMP_{len(self.comp_hash_to_uid):06d}"
            self.comp_hash_to_uid[h]   = uid
            self.comp_uid_to_hash[uid] = h
            self.comp_canonical[uid]   = canon
            self.comp_aliases[uid]     = {}

        uid = self.comp_hash_to_uid[h]
        if original_id:
            self.comp_aliases[uid][original_id] = source
        return uid

    # ── Registro de proteínas ─────────────────────────────────────────────────
    def get_protein_uid(
        self,
        sequence: str,
        original_id: str = "",
        source: str = ""
    ) -> str | None:
        """
        Retorna el UID interno de la proteína.
        Limpia caracteres no-aminoácido antes de hashear.
        """
        clean = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", sequence.upper().strip())
        if len(clean) < 10:
            return None

        h = self._hash(clean)
        if h not in self.prot_hash_to_uid:
            uid = f"PROT_{len(self.prot_hash_to_uid):06d}"
            self.prot_hash_to_uid[h]   = uid
            self.prot_uid_to_hash[uid] = h
            self.prot_canonical[uid]   = clean
            self.prot_aliases[uid]     = {}

        uid = self.prot_hash_to_uid[h]
        if original_id:
            self.prot_aliases[uid][original_id] = source
        return uid

    # ── Guardar / Cargar (JSON, sin pickle) ──────────────────────────────────
    def save(self):
        payload = {
            "version": "5.1.3",
            "n_compounds": len(self.comp_hash_to_uid),
            "n_proteins":  len(self.prot_hash_to_uid),
            "compounds": {
                "hash_to_uid":  self.comp_hash_to_uid,
                "uid_to_hash":  self.comp_uid_to_hash,
                "canonical":    self.comp_canonical,
                "aliases":      self.comp_aliases,
            },
            "proteins": {
                "hash_to_uid":  self.prot_hash_to_uid,
                "uid_to_hash":  self.prot_uid_to_hash,
                "canonical":    self.prot_canonical,
                "aliases":      self.prot_aliases,
            }
        }
        out = self.path / "mapping.json"
        out.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"💾 Registry guardado: {out}")
        print(f"   {payload['n_compounds']} compuestos únicos | {payload['n_proteins']} proteínas únicas")

    def load(self) -> bool:
        src = self.path / "mapping.json"
        if not src.exists():
            print("ℹ️  No existe registry previo — se creará uno nuevo.")
            return False
        payload = json.loads(src.read_text(encoding="utf-8"))
        self.comp_hash_to_uid = payload["compounds"]["hash_to_uid"]
        self.comp_uid_to_hash = payload["compounds"]["uid_to_hash"]
        self.comp_canonical   = payload["compounds"]["canonical"]
        self.comp_aliases     = payload["compounds"]["aliases"]
        self.prot_hash_to_uid = payload["proteins"]["hash_to_uid"]
        self.prot_uid_to_hash = payload["proteins"]["uid_to_hash"]
        self.prot_canonical   = payload["proteins"]["canonical"]
        self.prot_aliases     = payload["proteins"]["aliases"]
        print(f"✅ Registry cargado: {payload['n_compounds']} compuestos | {payload['n_proteins']} proteínas")
        return True

    def stats(self):
        print(f"\n📊 UnifiedRegistry stats:")
        print(f"   Compuestos únicos: {len(self.comp_hash_to_uid)}")
        print(f"   Proteínas únicas:  {len(self.prot_hash_to_uid)}")
        # Aliase cross-dataset
        multi_comp = sum(1 for v in self.comp_aliases.values() if len(v) > 1)
        multi_prot = sum(1 for v in self.prot_aliases.values() if len(v) > 1)
        print(f"   Compuestos en >1 dataset: {multi_comp}")
        print(f"   Proteínas en >1 dataset:  {multi_prot}")

    def find_similar_compounds(
        self,
        query_smiles: str,
        threshold: float = 0.85,
        max_results: int = 10
    ) -> list[dict]:
        """
        Busca compuestos similares en el registro por Tanimoto.
        Útil para saber si un compuesto nuevo ya tiene un equivalente.
        """
        results = []
        for uid, canon in self.comp_canonical.items():
            sim = self.tanimoto(query_smiles, canon)
            if sim >= threshold:
                results.append({"uid": uid, "tanimoto": sim,
                                 "canonical_smiles": canon,
                                 "aliases": self.comp_aliases.get(uid, {})})
        results.sort(key=lambda x: x["tanimoto"], reverse=True)
        return results[:max_results]

    def find_similar_proteins(
        self,
        query_seq: str,
        threshold: float = 0.90,
        max_results: int = 10
    ) -> list[dict]:
        """Busca proteínas similares por identidad de secuencia."""
        results = []
        for uid, canon in self.prot_canonical.items():
            sim = self.sequence_identity(query_seq, canon)
            if sim >= threshold:
                results.append({"uid": uid, "identity": sim,
                                 "aliases": self.prot_aliases.get(uid, {})})
        results.sort(key=lambda x: x["identity"], reverse=True)
        return results[:max_results]


print("✅ UnifiedRegistry definido.")


✅ UnifiedRegistry definido.


---
## 🧹 CELDA 3: DataStandardizer v2 + Consolidación

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 3: DataStandardizer v3 + Consolidación Universal
#
# Soporta:
#   1. Todos los datasets conocidos (Yamanishi, HetioNet, MOA, Davis, KIBA)
#   2. Actualización de datasets existentes (merge inteligente sin duplicados)
#   3. Dataset nuevo con formato desconocido → guía interactiva al usuario
# ═══════════════════════════════════════════════════════════════════════════════

# ── Catálogo de formatos conocidos ────────────────────────────────────────────
# Cada entrada describe exactamente cómo leer cada dataset:
#   drug_file, prot_file, inter_file:  nombre del archivo
#   drug_id_col, drug_smi_col:         columna de ID y SMILES en drug_file
#   prot_id_col, prot_seq_col:         columna de ID y secuencia en prot_file
#   inter_drug_col, inter_prot_col:    columna de drug/prot en inter_file
#   inter_sep:                         separador del inter_file
#   inter_header:                      True si la primera fila es header
#   id_prefix_sep:                     separador de prefijo ("::" → limpiar antes)
#   drug_id_format, prot_id_format:    descripción para logging

DATASET_CATALOG: dict[str, dict] = {
    "dti/yamanishi_08": {
        "task": "dti",
        "drug_file": "drug_smiles.csv",  "drug_id_col": "drug_id",  "drug_smi_col": "smiles",
        "prot_file": "protein_seq.csv",  "prot_id_col": "pro_id",   "prot_seq_col": "seq",
        "inter_file": "dti.csv",
        "inter_drug_col": 0, "inter_prot_col": 2,   # col1 = "DRUG_TARGET" (relación)
        "inter_sep": "\t",   "inter_header": False,
        "id_prefix_sep": None,            # IDs limpios: D00448, hsa:10
        "drug_id_format": "D#####",       "prot_id_format": "hsa:####",
        "n_splits": 10,
    },
    "dti/hetionet": {
        "task": "dti",
        "drug_file": "drug_smiles.csv",  "drug_id_col": "drug_id",  "drug_smi_col": "smiles",
        "prot_file": "protein_seq.csv",  "prot_id_col": "pro_id",   "prot_seq_col": "seq",
        "inter_file": "dti.csv",
        "inter_drug_col": 0, "inter_prot_col": 2,   # col1 = "Compound:Gene" (relación)
        "inter_sep": "\t",   "inter_header": False,
        "id_prefix_sep": "::",            # Compound::DB00514 → DB00514
        "drug_id_format": "Compound::DB#####", "prot_id_format": "Gene::####",
        "n_splits": 10,
    },
    "moa/activation": {
        "task": "moa",
        "drug_file": "drug_smi.csv",     "drug_id_col": "DrugID",   "drug_smi_col": "smi",
        "prot_file": "tar_seq.csv",      "prot_id_col": "TargetID", "prot_seq_col": "seq",
        "inter_file": "dti.csv",
        "inter_drug_col": 0, "inter_prot_col": 1,
        "inter_sep": "\t",   "inter_header": True,  # primera fila = DrugID, TargetID
        "id_prefix_sep": None,
        "drug_id_format": "D######",     "prot_id_format": "T#####",
        "n_splits": 5,
    },
    "moa/inhibition": {
        "task": "moa",
        "drug_file": "drug_smi.csv",     "drug_id_col": "DrugID",   "drug_smi_col": "smi",
        "prot_file": "tar_seq.csv",      "prot_id_col": "TargetID", "prot_seq_col": "seq",
        "inter_file": "dti.csv",
        "inter_drug_col": 0, "inter_prot_col": 1,
        "inter_sep": "\t",   "inter_header": True,
        "id_prefix_sep": None,
        "drug_id_format": "D######",     "prot_id_format": "T#####",
        "n_splits": 5,
    },
    "dta/davis": {
        "task": "dta",
        "drug_file": "ligands_can.txt",  "drug_id_col": "__key__",  "drug_smi_col": "__value__",
        "prot_file": "proteins.txt",     "prot_id_col": "__key__",  "prot_seq_col": "__value__",
        "inter_file": None,              # matriz Y en archivo binario separado
        "id_prefix_sep": None,
        "drug_id_format": "########",    "prot_id_format": "UniProt/name",
        "n_splits": 5,
    },
    "dta/kiba": {
        "task": "dta",
        "drug_file": "ligands_can.txt",  "drug_id_col": "__key__",  "drug_smi_col": "__value__",
        "prot_file": "proteins.txt",     "prot_id_col": "__key__",  "prot_seq_col": "__value__",
        "inter_file": None,
        "id_prefix_sep": None,
        "drug_id_format": "CHEMBL######", "prot_id_format": "UniProt",
        "n_splits": 5,
    },
}


class DataStandardizer:
    """
    Estandariza datasets al Golden Format usando el DATASET_CATALOG.

    Golden Format:
        drugs:    DataFrame [cid, uid, smi]
        proteins: DataFrame [pid, uid, seq]
        interactions: DataFrame [drug_uid, prot_uid, y, source]

    Para cada dataset del catálogo, sabe exactamente:
        - Qué archivo leer y qué columnas usar
        - Cómo limpiar prefijos de IDs (Gene::, Compound::, etc.)
        - Si el archivo de interacciones tiene header o no
        - Cuántos folds usar en K-Fold

    Para datasets nuevos/desconocidos, usa detect_and_register()
    que guía al usuario interactivamente.
    """

    def __init__(self, registry: "UnifiedRegistry"):
        self.registry = registry

    @staticmethod
    def _clean_id(raw_id: str, prefix_sep: str | None = "::") -> str:
        """Elimina prefijos tipo Compound::, Gene::, etc."""
        s = str(raw_id).strip()
        if prefix_sep and prefix_sep in s:
            return s.split(prefix_sep, 1)[-1]
        return s

    def _load_entities(
        self,
        base: Path,
        cfg: dict,
        entity: str   # "drug" | "prot"
    ) -> pd.DataFrame:
        """
        Carga el archivo de entidades (drugs o proteínas) y retorna
        DataFrame estandarizado con columnas [cid/pid, smi/seq].
        Soporta CSV/TSV y JSON (ligands_can.txt, proteins.txt).
        """
        if entity == "drug":
            fname, id_col, val_col = cfg["drug_file"], cfg["drug_id_col"], cfg["drug_smi_col"]
            out_id, out_val = "cid", "smi"
        else:
            fname, id_col, val_col = cfg["prot_file"], cfg["prot_id_col"], cfg["prot_seq_col"]
            out_id, out_val = "pid", "seq"

        fpath = base / fname
        if not fpath.exists():
            return pd.DataFrame(columns=[out_id, out_val])

        # ── JSON key-value (Davis/KIBA style) ────────────────────────────────
        if id_col == "__key__":
            data = json.loads(fpath.read_text())
            df = pd.DataFrame(list(data.items()), columns=[out_id, out_val])
        else:
            # ── CSV/TSV ───────────────────────────────────────────────────────
            sep = "\t" if fname.endswith(".csv") or fname.endswith(".tsv") else ","
            df  = pd.read_csv(fpath, sep=sep)
            df  = df.rename(columns={id_col: out_id, val_col: out_val})
            df  = df[[out_id, out_val]].copy()

        # Limpiar prefijos
        prefix_sep = cfg.get("id_prefix_sep")
        df[out_id] = df[out_id].apply(lambda x: self._clean_id(x, prefix_sep))
        return df

    def _load_interactions(self, base: Path, cfg: dict) -> pd.DataFrame | None:
        """
        Carga el archivo de interacciones y retorna DataFrame [cid, pid].
        Retorna None para DTA (la matriz Y se carga por separado).
        """
        if cfg["inter_file"] is None:
            return None

        fpath = base / cfg["inter_file"]
        if not fpath.exists():
            return None

        header = 0 if cfg["inter_header"] else None
        df     = pd.read_csv(fpath, sep=cfg["inter_sep"], header=header)

        dc = cfg["inter_drug_col"]
        pc = cfg["inter_prot_col"]
        prefix_sep = cfg.get("id_prefix_sep")

        result = pd.DataFrame()
        result["cid"] = df.iloc[:, dc].apply(lambda x: self._clean_id(x, prefix_sep))
        result["pid"] = df.iloc[:, pc].apply(lambda x: self._clean_id(x, prefix_sep))
        return result

    def register_dataset(
        self,
        dataset_key: str,
        force_reregister: bool = False
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame | None]:
        """
        Registra un dataset del catálogo en el UnifiedRegistry.

        Args:
            dataset_key:       clave en DATASET_CATALOG (ej. "dti/yamanishi_08")
            force_reregister:  si True, re-registra aunque ya existan alias

        Returns:
            (drugs_df, prots_df, interactions_df)
            donde drugs_df=[cid,uid,smi], prots_df=[pid,uid,seq]
            interactions_df=[drug_uid,prot_uid,y,source] o None para DTA
        """
        if dataset_key not in DATASET_CATALOG:
            raise KeyError(f"Dataset '{dataset_key}' no en catálogo. "
                           f"Usa detect_and_register() para datasets nuevos.")

        cfg  = DATASET_CATALOG[dataset_key]
        task = cfg["task"]
        ds   = dataset_key.split("/")[1]
        base = PROJECT_ROOT / "data" / task / ds

        print(f"\n📂 Registrando {dataset_key}...")

        # ── Cargar y registrar drugs ──────────────────────────────────────────
        raw_drugs = self._load_entities(base, cfg, "drug")
        drugs_out = []
        drug_skipped = 0
        for _, row in raw_drugs.iterrows():
            uid = self.registry.get_compound_uid(
                row["smi"], original_id=row["cid"], source=dataset_key
            )
            if uid:
                drugs_out.append({"cid": row["cid"], "uid": uid,
                                   "smi": self.registry.comp_canonical[uid]})
            else:
                drug_skipped += 1

        drugs_df = pd.DataFrame(drugs_out).drop_duplicates(subset=["uid"])
        print(f"   💊 {len(drugs_df)} compuestos únicos "
              f"({drug_skipped} SMILES inválidos descartados)")

        # ── Cargar y registrar proteínas ──────────────────────────────────────
        raw_prots = self._load_entities(base, cfg, "prot")
        prots_out = []
        prot_skipped = 0
        for _, row in raw_prots.iterrows():
            uid = self.registry.get_protein_uid(
                row["seq"], original_id=row["pid"], source=dataset_key
            )
            if uid:
                prots_out.append({"pid": row["pid"], "uid": uid,
                                   "seq": self.registry.prot_canonical[uid]})
            else:
                prot_skipped += 1

        prots_df = pd.DataFrame(prots_out).drop_duplicates(subset=["uid"])
        print(f"   🧬 {len(prots_df)} proteínas únicas "
              f"({prot_skipped} secuencias inválidas descartadas)")

        # ── Cargar interacciones y mapear a UIDs ──────────────────────────────
        raw_inter = self._load_interactions(base, cfg)
        if raw_inter is not None:
            cid_map  = dict(zip(drugs_df["cid"], drugs_df["uid"]))
            pid_map  = dict(zip(prots_df["pid"], prots_df["uid"]))

            inter_df = pd.DataFrame()
            inter_df["drug_uid"] = raw_inter["cid"].map(cid_map)
            inter_df["prot_uid"] = raw_inter["pid"].map(pid_map)
            inter_df["y"]        = 1
            inter_df["source"]   = dataset_key
            inter_df = inter_df.dropna(subset=["drug_uid", "prot_uid"])
            inter_df = inter_df.drop_duplicates()

            coverage = len(inter_df) / len(raw_inter) * 100
            print(f"   🔗 {len(inter_df)}/{len(raw_inter)} interacciones mapeadas "
                  f"({coverage:.1f}% cobertura)")

            if coverage < 50:
                print(f"   ⚠️  Cobertura baja. Posible desacoplamiento de IDs.")
                print(f"      drug_uid nulos: "
                      f"{inter_df['drug_uid'].isna().sum()}")
                print(f"      prot_uid nulos: "
                      f"{inter_df['prot_uid'].isna().sum()}")
        else:
            inter_df = None
            print(f"   📊 DTA: matriz de afinidad Y se carga por separado")

        return drugs_df, prots_df, inter_df

    def detect_and_register(
        self,
        dataset_path: str | Path,
        task: str | None = None,
        dataset_name: str | None = None,
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame | None]:
        """
        Registra un dataset NUEVO con formato desconocido.
        Detecta automáticamente la estructura y guía al usuario
        cuando no puede inferirla.

        Args:
            dataset_path: ruta a la carpeta del dataset
            task:         "dti" | "dta" | "moa" | None (se pregunta si falta)
            dataset_name: nombre para identificarlo (se infiere del path si falta)
        """
        base = Path(dataset_path)
        if not base.exists():
            raise FileNotFoundError(f"No existe: {base}")

        name = dataset_name or base.name
        if task is None:
            print(f"\n❓ ¿Qué tipo de tarea es '{name}'?")
            print("   [1] dti — Drug-Target Interaction (clasificación binaria)")
            print("   [2] dta — Drug-Target Affinity (regresión)")
            print("   [3] moa — Mechanism of Action (clasificación binaria)")
            choice = input("   Opción (1/2/3): ").strip()
            task = {"1": "dti", "2": "dta", "3": "moa"}.get(choice, "dti")

        dataset_key = f"{task}/{name}"
        print(f"\n🔍 Analizando estructura de {dataset_key}...")

        # ── Listar archivos disponibles ───────────────────────────────────────
        files = sorted([f.name for f in base.iterdir()
                        if f.is_file() and f.suffix in
                        [".csv", ".tsv", ".txt", ".json"]])
        print(f"   Archivos encontrados: {files}")

        # ── Detectar archivo de drugs ─────────────────────────────────────────
        drug_file = self._detect_file(base, files,
            keywords=["drug", "ligand", "compound", "smiles", "smi"],
            label="compuestos/drugs")

        drug_id_col, drug_smi_col = self._detect_columns(
            base / drug_file,
            id_keywords=["id", "drug", "compound", "cid"],
            val_keywords=["smi", "smiles", "canonical"],
            label_id="ID de compuesto",
            label_val="SMILES"
        ) if drug_file else (None, None)

        # ── Detectar archivo de proteínas ─────────────────────────────────────
        prot_file = self._detect_file(base, files,
            keywords=["prot", "protein", "target", "seq", "gene"],
            label="proteínas/targets")

        prot_id_col, prot_seq_col = self._detect_columns(
            base / prot_file,
            id_keywords=["id", "target", "prot", "gene", "pid"],
            val_keywords=["seq", "sequence", "fasta"],
            label_id="ID de proteína",
            label_val="secuencia AA"
        ) if prot_file else (None, None)

        # ── Detectar archivo de interacciones ─────────────────────────────────
        inter_file = self._detect_file(base, files,
            keywords=["dti", "interaction", "bind", "pair", "label"],
            label="interacciones",
            required=False)

        inter_cfg = {}
        if inter_file:
            inter_cfg = self._detect_interaction_format(
                base / inter_file, drug_id_col, prot_id_col
            )

        # ── Detectar prefijo de IDs ───────────────────────────────────────────
        prefix_sep = self._detect_prefix(base, drug_file, drug_id_col)

        # ── Construir cfg dinámico ────────────────────────────────────────────
        cfg = {
            "task": task,
            "drug_file": drug_file, "drug_id_col": drug_id_col,
            "drug_smi_col": drug_smi_col,
            "prot_file": prot_file, "prot_id_col": prot_id_col,
            "prot_seq_col": prot_seq_col,
            "inter_file": inter_file,
            "inter_drug_col": inter_cfg.get("drug_col", 0),
            "inter_prot_col": inter_cfg.get("prot_col", 1),
            "inter_sep": inter_cfg.get("sep", "\t"),
            "inter_header": inter_cfg.get("header", False),
            "id_prefix_sep": prefix_sep,
            "drug_id_format": "auto-detected",
            "prot_id_format": "auto-detected",
            "n_splits": 10 if task == "dti" else 5,
        }

        # ── Guardar en catálogo para usos futuros ─────────────────────────────
        DATASET_CATALOG[dataset_key] = cfg
        catalog_path = REGISTRY_PATH / "dataset_catalog.json"
        existing = json.loads(catalog_path.read_text()) if catalog_path.exists() else {}
        existing[dataset_key] = cfg
        catalog_path.write_text(json.dumps(existing, indent=2), encoding="utf-8")
        print(f"\n💾 Formato guardado en: {catalog_path.name}")
        print(f"   (En el futuro, este dataset se procesará automáticamente)")

        # ── Registrar con la cfg detectada ───────────────────────────────────
        return self.register_dataset(dataset_key)

    # ── Helpers de detección ──────────────────────────────────────────────────
    @staticmethod
    def _detect_file(
        base: Path,
        files: list,
        keywords: list,
        label: str,
        required: bool = True
    ) -> str | None:
        """Detecta el archivo más probable para una entidad."""
        candidates = [f for f in files
                      if any(kw in f.lower() for kw in keywords)]
        if len(candidates) == 1:
            print(f"   ✅ {label}: detectado → {candidates[0]}")
            return candidates[0]
        elif len(candidates) > 1:
            print(f"\n   ❓ Múltiples candidatos para {label}: {candidates}")
            for i, f in enumerate(candidates):
                print(f"      [{i}] {f}")
            if required:
                idx = input(f"   Selecciona índice: ").strip()
                return candidates[int(idx)]
            else:
                idx = input(f"   Selecciona índice (Enter para ninguno): ").strip()
                return candidates[int(idx)] if idx else None
        else:
            if required:
                print(f"\n   ❓ No se detectó archivo de {label}.")
                print(f"      Archivos disponibles: {files}")
                fname = input(f"   Nombre del archivo: ").strip()
                return fname if fname else None
            return None

    @staticmethod
    def _detect_columns(
        fpath: Path,
        id_keywords: list,
        val_keywords: list,
        label_id: str,
        label_val: str
    ) -> tuple[str, str]:
        """Detecta columnas de ID y valor en un CSV."""
        try:
            df   = pd.read_csv(fpath, sep="\t", nrows=2)
            cols = df.columns.tolist()
        except Exception:
            df   = pd.read_csv(fpath, nrows=2)
            cols = df.columns.tolist()

        id_col  = next((c for c in cols
                        if any(kw in c.lower() for kw in id_keywords)), None)
        val_col = next((c for c in cols
                        if any(kw in c.lower() for kw in val_keywords)), None)

        if id_col and val_col:
            print(f"   ✅ {label_id}: '{id_col}' | {label_val}: '{val_col}'")
            return id_col, val_col

        print(f"\n   ❓ Columnas en {fpath.name}: {cols}")
        if not id_col:
            print(f"   Ejemplo fila: {df.iloc[0].tolist()}")
            id_col = input(f"   Columna de {label_id}: ").strip()
        if not val_col:
            val_col = input(f"   Columna de {label_val}: ").strip()
        return id_col, val_col

    @staticmethod
    def _detect_interaction_format(
        fpath: Path,
        drug_id_col: str,
        prot_id_col: str
    ) -> dict:
        """Detecta el formato del archivo de interacciones."""
        # Intentar con y sin header
        df_header = pd.read_csv(fpath, sep="\t", nrows=3)
        df_nohead = pd.read_csv(fpath, sep="\t", header=None, nrows=3)

        # Si la primera fila parece header (tiene strings tipo DrugID/TargetID)
        first_row = df_nohead.iloc[0].tolist()
        has_header = any(
            str(v).lower() in ["drugid", "targetid", "drug_id", "prot_id",
                                "compound", "gene", "cid", "pid"]
            for v in first_row
        )

        df   = df_header if has_header else df_nohead
        cols = df.columns.tolist()

        # Detectar columnas de drug y prot
        drug_col = prot_col = None
        for i, col in enumerate(cols):
            col_str = str(col).lower()
            sample  = str(df.iloc[0, i]).lower()
            if (any(kw in col_str for kw in ["drug","compound","cid"]) or
                any(kw in sample  for kw in ["db","d0","chembl","compound"])):
                drug_col = i
            elif (any(kw in col_str for kw in ["target","prot","gene","pid"]) or
                  any(kw in sample  for kw in ["hsa","gene","t"])):
                prot_col = i

        if drug_col is None or prot_col is None:
            print(f"\n   ❓ {fpath.name} primeras filas:")
            print(df.to_string())
            drug_col = int(input("   Índice columna drug: ").strip())
            prot_col = int(input("   Índice columna prot: ").strip())

        print(f"   ✅ Interacciones: drug=col{drug_col}, "
              f"prot=col{prot_col}, header={has_header}")
        return {"drug_col": drug_col, "prot_col": prot_col,
                "sep": "\t", "header": has_header}

    @staticmethod
    def _detect_prefix(base: Path, drug_file: str, id_col: str) -> str | None:
        """Detecta si los IDs tienen prefijos tipo 'Compound::'."""
        if not drug_file:
            return None
        try:
            df = pd.read_csv(base / drug_file, sep="\t", nrows=1)
            sample = str(df[id_col].iloc[0])
            if "::" in sample:
                prefix = sample.split("::")[0] + "::"
                print(f"   ✅ Prefijo detectado: '{prefix}' → se eliminará automáticamente")
                return "::"
        except Exception:
            pass
        return None


# ── Inicializar ───────────────────────────────────────────────────────────────
print("🔄 Inicializando registro universal...")
print()

registry = UnifiedRegistry()

# Cargar catálogo extendido si existe (datasets nuevos añadidos previamente)
catalog_path = REGISTRY_PATH / "dataset_catalog.json"
if catalog_path.exists():
    external = json.loads(catalog_path.read_text())
    DATASET_CATALOG.update(external)
    print(f"📋 Catálogo extendido cargado: {len(external)} datasets adicionales")

registry.load()
std = DataStandardizer(registry)

# ── Registrar todos los datasets del catálogo ─────────────────────────────────
all_drugs, all_prots = [], []

for dataset_key in DATASET_CATALOG:
    task = DATASET_CATALOG[dataset_key]["task"]
    ds   = dataset_key.split("/")[1]
    base = PROJECT_ROOT / "data" / task / ds

    if not base.exists():
        print(f"⏭️  {dataset_key}: carpeta no encontrada, saltando")
        continue

    try:
        drugs_df, prots_df, _ = std.register_dataset(dataset_key)
        all_drugs.append(drugs_df)
        all_prots.append(prots_df)
    except Exception as e:
        print(f"⚠️  {dataset_key}: error → {e}")

# ── Maestros unificados ───────────────────────────────────────────────────────
master_drugs_df = (pd.concat(all_drugs)
                   .drop_duplicates(subset=["uid"])
                   .reset_index(drop=True))
master_prots_df = (pd.concat(all_prots)
                   .drop_duplicates(subset=["uid"])
                   .reset_index(drop=True))

registry.save()

print(f"\n{'═'*55}")
print(f"✅ Consolidación completa")
print(f"   💊 Compuestos únicos: {len(master_drugs_df)}")
print(f"   🧬 Proteínas únicas:  {len(master_prots_df)}")
registry.stats()

print(f"\n💡 Para añadir un dataset nuevo:")
print(f"   drugs_df, prots_df, inter_df = std.detect_and_register(")
print(f"       '../data/dti/mi_nuevo_dataset/',")
print(f"       task='dti', dataset_name='mi_nuevo_dataset'")
print(f"   )")


🔄 Inicializando registro universal...

✅ Registry cargado: 18272 compuestos | 18903 proteínas

📂 Registrando dti/yamanishi_08...


[08:08:39] WARNING: not removing hydrogen atom without neighbors
[08:08:39] WARNING: not removing hydrogen atom without neighbors
[08:08:39] WARNING: not removing hydrogen atom without neighbors


   💊 786 compuestos únicos (0 SMILES inválidos descartados)
   🧬 988 proteínas únicas (0 secuencias inválidas descartadas)
   🔗 5111/5127 interacciones mapeadas (99.7% cobertura)

📂 Registrando dti/hetionet...
   💊 1438 compuestos únicos (0 SMILES inválidos descartados)
   🧬 18800 proteínas únicas (0 secuencias inválidas descartadas)
   🔗 49797/49942 interacciones mapeadas (99.7% cobertura)

📂 Registrando moa/activation...
   💊 1379 compuestos únicos (0 SMILES inválidos descartados)
   🧬 277 proteínas únicas (0 secuencias inválidas descartadas)
   🔗 1800/1913 interacciones mapeadas (94.1% cobertura)

📂 Registrando moa/inhibition...


[08:08:44] WARNING: not removing hydrogen atom without neighbors
[08:08:44] WARNING: not removing hydrogen atom without neighbors


   💊 13623 compuestos únicos (0 SMILES inválidos descartados)
   🧬 1040 proteínas únicas (0 secuencias inválidas descartadas)
   🔗 19857/21055 interacciones mapeadas (94.3% cobertura)

📂 Registrando dta/davis...
   💊 68 compuestos únicos (0 SMILES inválidos descartados)
   🧬 379 proteínas únicas (0 secuencias inválidas descartadas)
   📊 DTA: matriz de afinidad Y se carga por separado

📂 Registrando dta/kiba...
   💊 2068 compuestos únicos (0 SMILES inválidos descartados)
   🧬 229 proteínas únicas (0 secuencias inválidas descartadas)
   📊 DTA: matriz de afinidad Y se carga por separado
💾 Registry guardado: /home/yosh/Documentos/TDA-Net/data/registry/mapping.json
   18272 compuestos únicos | 18903 proteínas únicas

═══════════════════════════════════════════════════════
✅ Consolidación completa
   💊 Compuestos únicos: 18272
   🧬 Proteínas únicas:  18903

📊 UnifiedRegistry stats:
   Compuestos únicos: 18272
   Proteínas únicas:  18903
   Compuestos en >1 dataset: 1269
   Proteínas en >1 da

---
## 💾 CELDA 4: Feature Store Seguro (HDF5 + SafeTensors)

In [4]:
class FeatureStore:
    """
    Almacén de embeddings sin pickle.

    Formato:
        embeddings: HDF5  — un dataset por UID, key = uid
        metadata:   JSON  — dimensiones, modelo usado, fecha, versión

    Ventajas vs pickle/dill:
        - HDF5 no ejecuta código al cargar (no hay RCE posible)
        - Acceso parcial: puedes cargar solo el embedding de COMP_000042
          sin cargar los miles restantes en RAM
        - Portable a R, MATLAB, Julia, C++
        - SafeTensors para interoperabilidad con HuggingFace/torch
    """

    def __init__(self, base_path: Path = MASTER_PATH):
        self.base = Path(base_path)

    def _h5_path(self, name: str) -> Path:
        return self.base / f"{name}.h5"

    def _meta_path(self, name: str) -> Path:
        return self.base / f"{name}_meta.json"

    # ── Guardar ──────────────────────────────────────────────────────────────
    def save(self, features: dict, name: str, metadata: dict | None = None):
        """
        Guarda embeddings en HDF5.

        Args:
            features: dict {uid: np.ndarray}
            name:     nombre del archivo (sin extensión), ej. 'comp_chemberta_medium'
            metadata: dict con info del modelo, fecha, etc.
        """
        h5_path = self._h5_path(name)
        with h5py.File(h5_path, "w") as f:
            f.attrs["version"] = "5.1.3"
            for uid, emb in features.items():
                f.create_dataset(uid, data=emb.astype(np.float32), compression="gzip")

        # Metadata en JSON
        meta = metadata or {}
        meta.update({
            "name": name,
            "n_entities": len(features),
            "embed_dim": next(iter(features.values())).shape[0] if features else 0,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "format": "HDF5/gzip",
        })
        self._meta_path(name).write_text(json.dumps(meta, indent=2), encoding="utf-8")

        size_mb = h5_path.stat().st_size / 1e6
        print(f"💾 Guardado: {h5_path.name}  ({size_mb:.1f} MB, {len(features)} entidades)")

    # ── Cargar completo ───────────────────────────────────────────────────────
    def load(self, name: str) -> dict:
        """Carga todos los embeddings como dict {uid: np.ndarray}."""
        h5_path = self._h5_path(name)
        if not h5_path.exists():
            raise FileNotFoundError(f"Feature store no encontrado: {h5_path}")
        features = {}
        with h5py.File(h5_path, "r") as f:
            for uid in f.keys():
                features[uid] = f[uid][:]
        meta = json.loads(self._meta_path(name).read_text()) if self._meta_path(name).exists() else {}
        print(f"✅ Cargado: {h5_path.name}  ({len(features)} entidades, dim={meta.get('embed_dim','?')})")
        return features

    # ── Cargar individual ─────────────────────────────────────────────────────
    def get(self, name: str, uid: str) -> np.ndarray | None:
        """Carga el embedding de una sola entidad (sin cargar todo en RAM)."""
        h5_path = self._h5_path(name)
        if not h5_path.exists():
            return None
        with h5py.File(h5_path, "r") as f:
            return f[uid][:] if uid in f else None

    # ── Migrar desde PKL/dill ─────────────────────────────────────────────────
    def migrate_from_pkl(self, pkl_path: str, name: str, uid_map: dict | None = None):
        """
        Convierte un archivo PKL/dill existente a HDF5.
        uid_map: dict {old_cid: new_uid} para remapear claves al registro unificado.
        Uso: feature_store.migrate_from_pkl('../data/master_features/MASTER_compound_features_chemberta_medium.pkl',
                                             'comp_chemberta_medium', uid_map=cid_to_uid)
        """
        import dill
        print(f"📂 Leyendo {pkl_path} ...")
        with open(pkl_path, "rb") as f:
            old_data = dill.load(f)

        new_data = {}
        remapped, missing = 0, 0
        for old_key, emb in old_data.items():
            if uid_map:
                uid = uid_map.get(old_key)
                if uid:
                    new_data[uid] = np.array(emb, dtype=np.float32)
                    remapped += 1
                else:
                    missing += 1
            else:
                new_data[str(old_key)] = np.array(emb, dtype=np.float32)

        self.save(new_data, name, metadata={"migrated_from": pkl_path})
        if uid_map:
            print(f"   ✅ Remapeados: {remapped} | Sin UID en registry: {missing}")

    def list_stores(self):
        """Lista todos los feature stores disponibles."""
        stores = list(self.base.glob("*.h5"))
        if not stores:
            print("ℹ️  No hay feature stores todavía.")
            return
        print("📦 Feature stores disponibles:")
        for s in sorted(stores):
            meta_path = s.with_name(s.stem + "_meta.json")
            if meta_path.exists():
                meta = json.loads(meta_path.read_text())
                print(f"   • {s.name:45s} {meta.get('n_entities','?'):6} entidades | "
                      f"dim={meta.get('embed_dim','?')} | {meta.get('saved_at','')}")
            else:
                size_mb = s.stat().st_size / 1e6
                print(f"   • {s.name:45s} {size_mb:.1f} MB")


feature_store = FeatureStore(MASTER_PATH)

print("✅ FeatureStore listo.")
feature_store.list_stores()

# ── Migración desde PKL si existen ────────────────────────────────────────────
print("\n🔄 Buscando archivos PKL para migrar a HDF5...")

# Mapeos cid→uid para migración
cid_to_uid = dict(zip(master_drugs_df["cid"], master_drugs_df["uid"]))
pid_to_uid = dict(zip(master_prots_df["pid"], master_prots_df["uid"]))

for pkl_name, h5_name, uid_map in [
    ("MASTER_compound_features_chemberta_medium.pkl", "comp_chemberta_medium", cid_to_uid),
    ("MASTER_compound_features_chemberta_light.pkl",  "comp_chemberta_light",  cid_to_uid),
    ("MASTER_protein_features_esm2_large.pkl",        "prot_esm2_large",       pid_to_uid),
    ("MASTER_protein_features_esm2_medium.pkl",       "prot_esm2_medium",      pid_to_uid),
    ("MASTER_protein_features_esm2_light.pkl",        "prot_esm2_light",       pid_to_uid),
]:
    pkl_path = MASTER_PATH / pkl_name
    h5_path  = MASTER_PATH / f"{h5_name}.h5"
    if pkl_path.exists() and not h5_path.exists():
        print(f"   Migrando {pkl_name} → {h5_name}.h5 ...")
        feature_store.migrate_from_pkl(str(pkl_path), h5_name, uid_map=uid_map)
    elif h5_path.exists():
        print(f"   ✅ Ya existe: {h5_name}.h5")
    else:
        print(f"   ⏭️  No encontrado: {pkl_name} (se generará al extraer features)")


✅ FeatureStore listo.
📦 Feature stores disponibles:
   • comp_chemberta_medium.h5                       18272 entidades | dim=384 | 2026-05-14 13:30:21
   • prot_esm2_large.h5                             18903 entidades | dim=1280 | 2026-05-14 13:52:52

🔄 Buscando archivos PKL para migrar a HDF5...
   ✅ Ya existe: comp_chemberta_medium.h5
   ⏭️  No encontrado: MASTER_compound_features_chemberta_light.pkl (se generará al extraer features)
   ✅ Ya existe: prot_esm2_large.h5
   ⏭️  No encontrado: MASTER_protein_features_esm2_medium.pkl (se generará al extraer features)
   ⏭️  No encontrado: MASTER_protein_features_esm2_light.pkl (se generará al extraer features)


---
## 🧪 CELDA 5: Extracción de Features de Compuestos (ChemBERTa)

In [6]:
from transformers import AutoTokenizer, AutoModel

def cal_comp_feat(
    data: pd.DataFrame,
    model_key: str = "chemberta_medium",
    batch_size: int = 64,
    pooling: str = "cls",
    uid_col: str = "uid"
) -> dict:
    """
    Extrae embeddings de compuestos con ChemBERTa o SELFormer.
    Retorna dict {uid: np.ndarray} usando UIDs del UnifiedRegistry.

    Args:
        data:      DataFrame con columnas [uid, smi] (ya estandarizado)
        model_key: clave del PLM_CATALOG
        batch_size: compuestos por batch
        pooling:   'cls' | 'mean'
        uid_col:   nombre de la columna con el UID interno
    """
    cfg          = PLM_CATALOG["compound"][model_key]
    model_name   = cfg["name"]
    input_format = cfg["input_format"]

    opcion = "B — SELFormer/SELFIES" if input_format == "selfies" else "A — ChemBERTa/SMILES"
    print(f"\n🧪 {model_name}  [{opcion}]")
    print(f"   {cfg['params']} params | VRAM {cfg['vram_req']} | dim={cfg['embed_dim']} | pooling={pooling}")

    # Preparar texto de entrada
    comp_data = data[[uid_col, "smi"]].dropna().copy()
    if input_format == "selfies":
        comp_data["text"] = comp_data["smi"].apply(UnifiedRegistry.smiles_to_selfies)
        comp_data = comp_data.dropna(subset=["text"])
    else:
        comp_data["text"] = comp_data["smi"]  # ya canonicalizado

    comp_data = comp_data.drop_duplicates(subset=[uid_col]).reset_index(drop=True)
    total = len(comp_data)

    if total == 0:
        print("❌ No quedan compuestos válidos.")
        return {}

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModel.from_pretrained(model_name)
    if DEVICE.type == "cuda":
        model = model.half()
    model = model.to(DEVICE).eval()

    comp_feat  = {}
    error_uids = []
    start_time = time.time()
    ICONS = 30

    for bs in range(0, total, batch_size):
        batch    = comp_data.iloc[bs : bs + batch_size]
        uids     = batch[uid_col].tolist()
        texts    = batch["text"].tolist()

        try:
            enc = tokenizer(texts, padding=True, truncation=True,
                            max_length=512, return_tensors="pt")
            with torch.no_grad():
                out = model(input_ids=enc["input_ids"].to(DEVICE),
                            attention_mask=enc["attention_mask"].to(DEVICE))
            h = out.last_hidden_state
            if pooling == "cls":
                embs = h[:, 0, :]
            else:
                mask = enc["attention_mask"].unsqueeze(-1).float().to(DEVICE)
                embs = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

            for uid, emb in zip(uids, embs.cpu().float().numpy()):
                comp_feat[uid] = emb
        except Exception:
            error_uids.extend(uids)
        finally:
            torch.cuda.empty_cache()

        cur   = min(bs + batch_size, total)
        ratio = cur / total
        bar   = ("🧬" if input_format=="selfies" else "💊") * ceil(ratio*ICONS) + "🧪" * (ICONS - ceil(ratio*ICONS))
        el    = time.time() - start_time
        print(f"\r{int(ratio*100)}% |{bar}| {cur}/{total} [{el:.0f}s]", end="")

    model.cpu(); del model, tokenizer; torch.cuda.empty_cache()
    print(f"\n✅ {len(comp_feat)}/{total} procesados en {time.time()-start_time:.1f}s")
    if error_uids:
        print(f"⚠️  {len(error_uids)} errores")
    return comp_feat


# ── Ejecutar y guardar ────────────────────────────────────────────────────────
COMPOUND_MODEL_KEY = "chemberta_medium"   # ← Cambiar aquí

master_comp_feat = cal_comp_feat(
    master_drugs_df,
    model_key=COMPOUND_MODEL_KEY,
    pooling="cls"
)

feature_store.save(
    master_comp_feat,
    name=f"comp_{COMPOUND_MODEL_KEY}",
    metadata={
        "model": PLM_CATALOG["compound"][COMPOUND_MODEL_KEY]["name"],
        "model_key": COMPOUND_MODEL_KEY,
        "pooling": "cls",
        "n_datasets": "yamanishi+hetionet+moa+davis+kiba",
    }
)



🧪 DeepChem/ChemBERTa-77M-MTR  [A — ChemBERTa/SMILES]
   77M params | VRAM ~3GB | dim=768 | pooling=cls


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


100% |💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊💊| 18272/18272 [5s]
✅ 18272/18272 procesados en 5.3s
💾 Guardado: comp_chemberta_medium.h5  (82.5 MB, 18272 entidades)


---
## 🧬 CELDA 6: Extracción de Features de Proteínas (ESM-2 via HuggingFace)

In [7]:
# ⚠ ESM-2 se carga via HuggingFace transformers (NO fair-esm — deprecado)
# El nombre del modelo cambia: "esm2_t33_650M_UR50D" → "facebook/esm2_t33_650M_UR50D"
from transformers import AutoTokenizer, AutoModel as HFAutoModel

def cal_prot_feat(
    data: pd.DataFrame,
    model_key: str = "large",
    batch_size: int = 8,
    max_seq_len: int = 1022,
    uid_col: str = "uid"
) -> dict:
    """
    Genera embeddings de proteínas con ESM-2 vía HuggingFace.
    Retorna dict {uid: np.ndarray}.

    Args:
        data:       DataFrame con columnas [uid, seq]
        model_key:  'light' | 'medium' | 'large'
        batch_size: proteínas por batch (bajar si OOM)
        max_seq_len: truncar secuencias más largas
    """
    cfg        = PLM_CATALOG["protein"][model_key]
    model_name = cfg["name"]  # e.g. 'facebook/esm2_t33_650M_UR50D'

    print(f"\n🧬 ESM-2: {model_name}")
    print(f"   {cfg['params']} params | VRAM {cfg['vram_req']} | dim={cfg['embed_dim']}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = HFAutoModel.from_pretrained(model_name)
    if DEVICE.type == "cuda":
        model = model.half()
    model = model.to(DEVICE).eval()

    prot_data = data[[uid_col, "seq"]].drop_duplicates(subset=[uid_col]).copy()
    prot_data["seq"] = prot_data["seq"].str[:max_seq_len]
    prot_data = prot_data.reset_index(drop=True)
    total      = len(prot_data)
    prot_feat  = {}
    start_time = time.time()
    ICONS = 25

    for bs in range(0, total, batch_size):
        batch = prot_data.iloc[bs : bs + batch_size]
        uids  = batch[uid_col].tolist()
        seqs  = batch["seq"].tolist()

        enc = tokenizer(seqs, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_seq_len + 2)
        with torch.no_grad():
            out = model(input_ids=enc["input_ids"].to(DEVICE),
                        attention_mask=enc["attention_mask"].to(DEVICE))

        # Mean pooling sobre tokens de residuos (excluir [CLS] y [EOS])
        hidden = out.last_hidden_state   # (B, L, dim)
        mask   = enc["attention_mask"].unsqueeze(-1).float().to(DEVICE)
        embs   = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        for uid, emb in zip(uids, embs.cpu().float().numpy()):
            prot_feat[uid] = emb

        del out, hidden, mask, embs, enc
        torch.cuda.empty_cache()

        cur   = min(bs + batch_size, total)
        ratio = cur / total
        filled = ceil(ratio * ICONS)
        bar   = "🧬" * filled + "🔬" * (ICONS - filled)
        el    = time.time() - start_time
        eta   = (total - cur) / (cur / el) if cur > 0 else 0
        print(f"\r{int(ratio*100)}% |{bar}| {cur}/{total} [ETA: {eta:.0f}s]", end="")

    model.cpu(); del model, tokenizer; torch.cuda.empty_cache()
    print(f"\n✅ {len(prot_feat)}/{total} proteínas en {time.time()-start_time:.1f}s")
    return prot_feat


# ── Ejecutar y guardar ────────────────────────────────────────────────────────
PROTEIN_MODEL_KEY = "large"   # ← Cambiar aquí

master_prot_feat = cal_prot_feat(
    master_prots_df,
    model_key=PROTEIN_MODEL_KEY
)

feature_store.save(
    master_prot_feat,
    name=f"prot_esm2_{PROTEIN_MODEL_KEY}",
    metadata={
        "model": PLM_CATALOG["protein"][PROTEIN_MODEL_KEY]["name"],
        "model_key": PROTEIN_MODEL_KEY,
        "backend": "huggingface_transformers",
    }
)



🧬 ESM-2: facebook/esm2_t33_650M_UR50D
   650M params | VRAM ~8GB | dim=1280


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


100% |🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬🧬| 18903/18903 [ETA: 0s]]
✅ 18903/18903 proteínas en 1343.5s
💾 Guardado: prot_esm2_large.h5  (137.1 MB, 18903 entidades)


---
## 🔀 CELDA 7: Splits K-Fold con UIDs

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELDA 7: Splits K-Fold — lee del DATASET_CATALOG (sin hardcodear nada)
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.model_selection import KFold, GroupKFold

def split_dti(dataset: str, task: str = "dti", n_splits: int | None = None) -> None:
    """
    Genera folds K-Fold para DTI/MOA.
    Lee el formato correcto del DATASET_CATALOG — sin asumir columnas.

    Args:
        dataset:  nombre del dataset (ej. "yamanishi_08", "hetionet")
        task:     "dti" | "moa"
        n_splits: número de folds (None = usar el del catálogo)
    """
    dataset_key = f"{task}/{dataset}"
    if dataset_key not in DATASET_CATALOG:
        raise KeyError(f"'{dataset_key}' no en catálogo. "
                       f"Usa std.detect_and_register() primero.")

    cfg    = DATASET_CATALOG[dataset_key]
    k      = n_splits or cfg.get("n_splits", 10)
    base   = PROJECT_ROOT / "data" / task / dataset

    # ── Leer interacciones usando el catálogo ─────────────────────────────────
    inter_file = cfg["inter_file"]
    if not inter_file or not (base / inter_file).exists():
        print(f"⚠️  {dataset_key}: sin archivo de interacciones, saltando splits")
        return

    header = 0 if cfg["inter_header"] else None
    raw    = pd.read_csv(base / inter_file, sep=cfg["inter_sep"], header=header)

    dc         = cfg["inter_drug_col"]
    pc         = cfg["inter_prot_col"]
    prefix_sep = cfg.get("id_prefix_sep")

    raw_cids = raw.iloc[:, dc].apply(
        lambda x: DataStandardizer._clean_id(x, prefix_sep))
    raw_pids = raw.iloc[:, pc].apply(
        lambda x: DataStandardizer._clean_id(x, prefix_sep))

    # ── Mapear a UIDs ─────────────────────────────────────────────────────────
    cid_map = dict(zip(master_drugs_df["cid"], master_drugs_df["uid"]))
    pid_map = dict(zip(master_prots_df["pid"], master_prots_df["uid"]))
    # Incluir UIDs directos (por si el archivo ya usa UIDs)
    cid_map.update({v: v for v in master_drugs_df["uid"]})
    pid_map.update({v: v for v in master_prots_df["uid"]})

    drug_uids = raw_cids.map(cid_map)
    prot_uids = raw_pids.map(pid_map)

    mask = drug_uids.notna() & prot_uids.notna()
    coverage = mask.sum() / len(raw) * 100
    print(f"\n🔀 {dataset_key}: {mask.sum()}/{len(raw)} pares válidos ({coverage:.1f}%)")

    if mask.sum() == 0:
        print(f"❌ Cobertura 0% — verifica el registro del dataset")
        print(f"   cid sample en archivo: {raw_cids.iloc[:3].tolist()}")
        print(f"   cid sample en registry: {list(cid_map.keys())[:3]}")
        return

    if coverage < 50:
        print(f"⚠️  Cobertura baja ({coverage:.1f}%). "
              f"Considera re-registrar con std.register_dataset('{dataset_key}')")

    # ── Construir positivos y negativos ───────────────────────────────────────
    pos_df = pd.DataFrame({
        "drug_uid": drug_uids[mask].values,
        "prot_uid": prot_uids[mask].values,
        "y":        1,
        "source":   dataset_key
    }).drop_duplicates(subset=["drug_uid", "prot_uid"])

    all_d   = pos_df["drug_uid"].unique()
    all_p   = pos_df["prot_uid"].unique()
    pos_set = set(zip(pos_df["drug_uid"], pos_df["prot_uid"]))

    neg_pairs, attempts = [], 0
    rng = np.random.default_rng(42)
    target = len(pos_df)
    while len(neg_pairs) < target and attempts < target * 10:
        d = rng.choice(all_d)
        p = rng.choice(all_p)
        if (d, p) not in pos_set:
            neg_pairs.append((d, p))
            pos_set.add((d, p))
        attempts += 1

    neg_df = pd.DataFrame(neg_pairs, columns=["drug_uid", "prot_uid"])
    neg_df["y"]      = 0
    neg_df["source"] = dataset_key

    full_df = pd.concat([pos_df, neg_df]).reset_index(drop=True)
    print(f"   Positivos: {len(pos_df)} | Negativos: {len(neg_df)} | "
          f"Total: {len(full_df)}")

    # ── K-Fold splits ─────────────────────────────────────────────────────────
    for setting, group_col in [
        ("warm_start",        None),
        ("drug_coldstart",    "drug_uid"),
        ("protein_coldstart", "prot_uid"),
    ]:
        save_path = base / "data_folds" / setting
        save_path.mkdir(parents=True, exist_ok=True)

        actual_k = min(k, len(full_df))
        if actual_k < k:
            print(f"   ⚠️  {setting}: reduciendo folds a {actual_k} "
                  f"(solo {len(full_df)} muestras)")

        if group_col is None:
            kf     = KFold(n_splits=actual_k, shuffle=True, random_state=42)
            splits = list(kf.split(full_df))
        else:
            n_groups = full_df[group_col].nunique()
            actual_k = min(actual_k, n_groups)
            gkf      = GroupKFold(n_splits=actual_k)
            splits   = list(gkf.split(full_df, groups=full_df[group_col]))

        for i, (tr_idx, te_idx) in enumerate(splits):
            full_df.iloc[tr_idx].to_csv(
                save_path / f"fold_{i}_train.csv", index=False, sep="\t")
            full_df.iloc[te_idx].to_csv(
                save_path / f"fold_{i}_test.csv",  index=False, sep="\t")

        print(f"   ✅ {setting}: {len(splits)} folds guardados")


def split_dta(dataset: str, n_splits: int | None = None) -> None:
    """
    Genera folds K-Fold para DTA (Davis/KIBA).
    Lee la matriz de afinidad Y y mapea a UIDs del registry.
    """
    dataset_key = f"dta/{dataset}"
    cfg  = DATASET_CATALOG.get(dataset_key, {})
    k    = n_splits or cfg.get("n_splits", 5)
    base = PROJECT_ROOT / "data" / "dta" / dataset

    ligands  = json.loads((base / "ligands_can.txt").read_text())
    proteins = json.loads((base / "proteins.txt").read_text())
    affinity = pickle.load(open(base / "Y", "rb"), encoding="latin1")

    cid_map = dict(zip(master_drugs_df["cid"], master_drugs_df["uid"]))
    pid_map = dict(zip(master_prots_df["pid"], master_prots_df["uid"]))
    cid_map.update({v: v for v in master_drugs_df["uid"]})
    pid_map.update({v: v for v in master_prots_df["uid"]})

    drug_ids   = list(ligands.keys())
    target_ids = list(proteins.keys())

    rows = []
    for di, cid in enumerate(drug_ids):
        for ti, pid in enumerate(target_ids):
            val = affinity[di, ti]
            if not np.isnan(val):
                duid = cid_map.get(cid)
                puid = pid_map.get(pid)
                if duid and puid:
                    rows.append({"drug_uid": duid, "prot_uid": puid,
                                 "y": val, "source": dataset_key})

    full_df = pd.DataFrame(rows)
    print(f"\n🔀 {dataset_key}: {len(full_df)} pares con afinidad conocida")

    if len(full_df) == 0:
        print(f"❌ Sin pares válidos — verifica que Davis/KIBA estén en el registry")
        return

    for setting, group_col in [
        ("warm_start",        None),
        ("drug_coldstart",    "drug_uid"),
        ("protein_coldstart", "prot_uid"),
    ]:
        save_path = base / "data_folds" / setting
        save_path.mkdir(parents=True, exist_ok=True)

        actual_k = min(k, len(full_df))
        if group_col is None:
            kf     = KFold(n_splits=actual_k, shuffle=True, random_state=42)
            splits = list(kf.split(full_df))
        else:
            n_groups = full_df[group_col].nunique()
            actual_k = min(actual_k, n_groups)
            gkf      = GroupKFold(n_splits=actual_k)
            splits   = list(gkf.split(full_df, groups=full_df[group_col]))

        for i, (tr_idx, te_idx) in enumerate(splits):
            full_df.iloc[tr_idx].to_csv(
                save_path / f"fold_{i}_train.csv", index=False, sep="\t")
            full_df.iloc[te_idx].to_csv(
                save_path / f"fold_{i}_test.csv",  index=False, sep="\t")

        print(f"   ✅ {setting}: {len(splits)} folds guardados")


print("✅ split_dti() y split_dta() listos — leen del DATASET_CATALOG")
print()
print("Uso:")
print("  split_dti('yamanishi_08', task='dti')")
print("  split_dti('hetionet',     task='dti')")
print("  split_dti('activation',   task='moa')")
print("  split_dti('inhibition',   task='moa')")
print("  split_dta('davis')")
print("  split_dta('kiba')")
split_dti('yamanishi_08', task='dti')
split_dti('hetionet',     task='dti')

✅ split_dti() y split_dta() listos — leen del DATASET_CATALOG

Uso:
  split_dti('yamanishi_08', task='dti')
  split_dti('hetionet',     task='dti')
  split_dti('activation',   task='moa')
  split_dti('inhibition',   task='moa')
  split_dta('davis')
  split_dta('kiba')

🔀 dti/yamanishi_08: 5111/5127 pares válidos (99.7%)
   Positivos: 5111 | Negativos: 5111 | Total: 10222
   ✅ warm_start: 10 folds guardados
   ✅ drug_coldstart: 10 folds guardados
   ✅ protein_coldstart: 10 folds guardados

🔀 dti/hetionet: 26536/49942 pares válidos (53.1%)
   Positivos: 26536 | Negativos: 26536 | Total: 53072
   ✅ warm_start: 10 folds guardados
   ✅ drug_coldstart: 10 folds guardados
   ✅ protein_coldstart: 10 folds guardados


---
## 🤖 CELDA 8: Entrenamiento K-Fold con AutoGluon

In [6]:
from autogluon.tabular import TabularPredictor
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy import stats

# ── Métricas ─────────────────────────────────────────────────────────────────
def rmse(y_true, y_pred):    return float(np.sqrt(np.mean((y_true - y_pred)**2)))
def mse(y_true, y_pred):     return float(np.mean((y_true - y_pred)**2))
def pearson(y_true, y_pred): return float(stats.pearsonr(y_true, y_pred)[0])
def spearman(y,p):           return float(stats.spearmanr(y, p)[0])
def ci(y_true, y_pred):
    n = len(y_true); total = concordant = 0
    for i in range(n):
        for j in range(i+1, n):
            if y_true[i] != y_true[j]:
                total += 1
                if (y_pred[i] > y_pred[j]) == (y_true[i] > y_true[j]): concordant += 1
    return concordant / total if total > 0 else 0.5


def load_fold_data(folds_path: Path, fold_idx: int,
                   comp_feat: dict, prot_feat: dict) -> tuple:
    """
    Carga un fold y construye el feature vector concatenado.
    Robusto a IDs sin features (bug hetionet corregido):
    - Omite pares sin embedding con warning
    - Infiere dimensiones del primer par válido (no de la última iteración)
    - Lanza error claro si el fold queda vacío
    """
    train_df = pd.read_csv(folds_path / f"fold_{fold_idx}_train.csv", sep="\t")
    test_df  = pd.read_csv(folds_path / f"fold_{fold_idx}_test.csv",  sep="\t")

    def build_features(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
        rows, skipped = [], 0

        # Inferir dimensiones del primer par válido
        n_comp = n_prot = None
        for _, row in df.iterrows():
            cf = comp_feat.get(row["drug_uid"])
            pf = prot_feat.get(row["prot_uid"])
            if cf is not None and pf is not None:
                n_comp, n_prot = len(cf), len(pf)
                break

        if n_comp is None:
            raise ValueError(
                f"❌ Fold {fold_idx} ({split_name}): ningún par tiene features.\n"
                f"   drug_uids sample: {df['drug_uid'].iloc[:3].tolist()}\n"
                f"   prot_uids sample: {df['prot_uid'].iloc[:3].tolist()}\n"
                f"   Verifica que los splits usen UIDs y no IDs originales."
            )

        for _, row in df.iterrows():
            cf = comp_feat.get(row["drug_uid"])
            pf = prot_feat.get(row["prot_uid"])
            if cf is not None and pf is not None:
                rows.append(np.concatenate([cf, pf, [row["y"]]]))
            else:
                skipped += 1

        if skipped > 0:
            print(f"   ⚠️  {split_name}: {skipped}/{len(df)} pares omitidos (sin features)")

        cols = [f"c{i}" for i in range(n_comp)] + [f"p{i}" for i in range(n_prot)] + ["y"]
        return pd.DataFrame(rows, columns=cols)

    return build_features(train_df, "train"), build_features(test_df, "test")


def kfold_validation(
    task: str,
    dataset: str,
    setting: str,
    comp_model_key: str = "chemberta_medium",
    prot_model_key: str = "large",
    preset: str = "good_quality",
    ex_model: list = [],
    num_gpus: int = 1,
) -> pd.DataFrame:
    """
    Validación K-Fold completa. Lee features desde HDF5, guarda bundle JSON.

    Args:
        task:           'dti' | 'dta' | 'moa'
        dataset:        nombre del dataset
        setting:        'warm_start' | 'drug_coldstart' | 'protein_coldstart'
        comp_model_key: clave del PLM_CATALOG para compuestos
        prot_model_key: clave del PLM_CATALOG para proteínas
        preset:         AutoGluon preset ('best_quality' | 'good_quality' | 'fast')
        ex_model:       modelos a excluir de AutoGluon
        num_gpus:       1 para habilitar CatBoost+NN_TORCH en GPU
    """
    print(f"\n🚀 DTIAM V5-1.3: {task.upper()} | {dataset} | {setting}")
    comp_params = PLM_CATALOG["compound"][comp_model_key]["params"]
    prot_params = PLM_CATALOG["protein"][prot_model_key]["params"]
    print(f"   PLMs: ChemBERTa-{comp_params} + ESM-2-{prot_params}")

    task_cfg = {
        "dti": {"base": Path(f"../data/dti/{dataset}/"), "k": 10,
                "metric": "roc_auc", "cols": ["AUROC","AUPR"]},
        "dta": {"base": Path(f"../data/dta/{dataset}/"), "k": 5,
                "metric": "root_mean_squared_error",
                "cols": ["RMSE","MSE","Pearson","Spearman","CI"]},
        "moa": {"base": Path(f"../data/moa/{dataset}/"), "k": 5,
                "metric": "roc_auc", "cols": ["AUROC","AUPR"]},
    }[task]

    # ── Cargar features desde HDF5 ────────────────────────────────────────────
    comp_store_name = f"comp_{comp_model_key}"
    prot_store_name = f"prot_esm2_{prot_model_key}"
    print(f"📥 Cargando features: {comp_store_name}.h5, {prot_store_name}.h5")
    comp_feat = feature_store.load(comp_store_name)
    prot_feat = feature_store.load(prot_store_name)

    folds_path = task_cfg["base"] / "data_folds" / setting
    res_all    = pd.DataFrame(columns=task_cfg["cols"])
    start_time = time.time()

    for i in range(task_cfg["k"]):
        fold_start = time.time()
        print(f"\n🔄 Fold {i+1}/{task_cfg['k']}...", end=" ")

        train_data, test_data = load_fold_data(folds_path, i, comp_feat, prot_feat)
        test_nolab = test_data.drop(columns=["y"])

        fold_model_dir = str(MODELS_PATH / f"{task}_{dataset}_{setting}_fold{i}")

        predictor = TabularPredictor(
            label="y", eval_metric=task_cfg["metric"],
            path=fold_model_dir, verbosity=0
        ).fit(
            train_data=train_data,
            excluded_model_types=ex_model,
            presets=preset,
            num_gpus=num_gpus,
            ag_args_fit={"num_gpus": num_gpus},
            hyperparameters={
                "CAT": {"task_type": "GPU"} if num_gpus > 0 else {},
                "NN_TORCH": {},
            } if num_gpus > 0 else None,
        )

        if task == "dta":
            P   = np.array(predictor.predict(test_nolab))
            G   = np.array(test_data["y"])
            ret = [rmse(G,P), mse(G,P), pearson(G,P), spearman(G,P), ci(G,P)]
            print(f"RMSE={ret[0]:.4f} | CI={ret[4]:.4f}")
            res_all.loc[i] = ret
        else:
            probs  = predictor.predict_proba(test_nolab)
            probs1 = probs.iloc[:, 1] if isinstance(probs, pd.DataFrame) else probs
            y_true = np.array(test_data["y"])
            auroc  = roc_auc_score(y_true, np.array(probs1))
            aupr   = average_precision_score(y_true, np.array(probs1))
            print(f"AUROC={auroc:.4f} | AUPR={aupr:.4f}")
            res_all.loc[i] = [auroc, aupr]

        print(f"   ⏱️ {time.time()-fold_start:.1f}s")

    # ── Resumen ───────────────────────────────────────────────────────────────
    print("\n" + "="*55)
    print("📊 RESULTADOS FINALES (mean ± std)")
    print("="*55)
    summary = pd.DataFrame({"Mean": res_all.mean(), "Std": res_all.std()})
    print(summary.to_string())
    elapsed_min = (time.time() - start_time) / 60
    print(f"\n⏱️ Tiempo total: {elapsed_min:.1f} min")

    # ── Guardar resultados CSV ────────────────────────────────────────────────
    out_csv = RESULTS_PATH / f"{task}_{dataset}_{setting}_{comp_model_key}_{prot_model_key}.csv"
    res_all.to_csv(out_csv, index=True, sep="\t")

    # ── Guardar bundle JSON (sin pickle) ──────────────────────────────────────
    bundle = {
        "version": "5.1.3",
        "task": task, "dataset": dataset, "setting": setting,
        "comp_model_key": comp_model_key, "prot_model_key": prot_model_key,
        "comp_store": comp_store_name, "prot_store": prot_store_name,
        "preset": preset, "n_folds": task_cfg["k"],
        "fold_dirs": [
            str(MODELS_PATH / f"{task}_{dataset}_{setting}_fold{i}")
            for i in range(task_cfg["k"])
        ],
        "best_fold": int(res_all["AUROC"].idxmax() if "AUROC" in res_all else res_all["RMSE"].idxmin()),
        "metrics_mean": summary["Mean"].to_dict(),
        "metrics_std":  summary["Std"].to_dict(),
        "elapsed_min": round(elapsed_min, 1),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    bundle_path = RESULTS_PATH / f"bundle_{task}_{dataset}_{setting}.json"
    bundle_path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")

    print(f"💾 CSV:    {out_csv.name}")
    print(f"📦 Bundle: {bundle_path.name}")
    return res_all


print("✅ kfold_validation() listo.")
print('   Ejemplo: kfold_validation("dti", "yamanishi_08", "warm_start")')

#kfold_validation("dti", "yamanishi_08", "warm_start")
#kfold_validation("dti", "hetionet", "warm_start")


✅ kfold_validation() listo.
   Ejemplo: kfold_validation("dti", "yamanishi_08", "warm_start")


In [7]:
import psutil, ray

# Ver cuánta RAM tienes disponible ahora mismo
ram = psutil.virtual_memory()
print(f"RAM total:      {ram.total/1e9:.1f} GB")
print(f"RAM disponible: {ram.available/1e9:.1f} GB")
print(f"RAM usada:      {ram.used/1e9:.1f} GB ({ram.percent:.0f}%)")

# Matar Ray si quedó zombie de la corrida anterior
if ray.is_initialized():
    ray.shutdown()
    print("✅ Ray detenido")

# Ver si quedan procesos ray zombie
import subprocess
result = subprocess.run(["pgrep", "-c", "ray"], capture_output=True, text=True)
n_ray = result.stdout.strip()
print(f"Procesos ray activos: {n_ray}")
if int(n_ray or 0) > 0:
    subprocess.run(["ray", "stop", "--force"])
    print("✅ Ray forzado a detenerse")

RAM total:      33.6 GB
RAM disponible: 19.2 GB
RAM usada:      14.4 GB (43%)
Procesos ray activos: 0


In [8]:
import ray, psutil

ray.init(
    num_cpus             = 4,      # máximo 4 workers paralelos (no todos los cores)
    object_store_memory  = int(8e9),  # 8GB para el object store de Ray
    ignore_reinit_error  = True,
    log_to_driver        = False,  # silencia los mensajes de raylet
    configure_logging    = False,
)

ram = psutil.virtual_memory()
print(f"✅ Ray iniciado")
print(f"   RAM disponible: {ram.available/1e9:.1f} GB")
print(f"   Workers máximos: 4")
print(f"   Object store: 8 GB")

✅ Ray iniciado
   RAM disponible: 15.9 GB
   Workers máximos: 4
   Object store: 8 GB


In [9]:
# ── PASO 2: Reducir dimensiones (de 2048 a 512) ───────────────────────────────
from sklearn.decomposition import PCA
import numpy as np

print("🔧 Cargando features y reduciendo dimensiones...")

raw_comp = feature_store.load("comp_chemberta_medium")
raw_prot = feature_store.load("prot_esm2_large")

# PCA compounds: 768 → 256
cids  = list(raw_comp.keys())
C     = np.stack([raw_comp[c] for c in cids])
pca_c = PCA(n_components=256, random_state=42)
C_r   = pca_c.fit_transform(C)
comp_feat_red = {cid: C_r[i] for i, cid in enumerate(cids)}
print(f"💊 Compounds: {C.shape[1]}d → 256d "
      f"| varianza retenida: {pca_c.explained_variance_ratio_.sum():.4f}")

# PCA proteins: 1280 → 256
pids  = list(raw_prot.keys())
P     = np.stack([raw_prot[p] for p in pids])
pca_p = PCA(n_components=256, random_state=42)
P_r   = pca_p.fit_transform(P)
prot_feat_red = {pid: P_r[i] for i, pid in enumerate(pids)}
print(f"🧬 Proteins:  {P.shape[1]}d → 256d "
      f"| varianza retenida: {pca_p.explained_variance_ratio_.sum():.4f}")

# Memoria liberada
del C, C_r, P, P_r
import gc; gc.collect()

ram = psutil.virtual_memory()
print(f"\n✅ Features reducidos — RAM disponible ahora: {ram.available/1e9:.1f} GB")
print(f"   Dimensión final por par: 512 (256 comp + 256 prot)")

🔧 Cargando features y reduciendo dimensiones...
✅ Cargado: comp_chemberta_medium.h5  (18272 entidades, dim=384)
✅ Cargado: prot_esm2_large.h5  (18903 entidades, dim=1280)
💊 Compounds: 384d → 256d | varianza retenida: 0.9985
🧬 Proteins:  1280d → 256d | varianza retenida: 0.9626

✅ Features reducidos — RAM disponible ahora: 15.2 GB
   Dimensión final por par: 512 (256 comp + 256 prot)


In [10]:
# ── PASO 3: kfold_validation con configuración conservadora ──────────────────
# Pega esto ANTES de llamar kfold_validation — redefine solo el .fit() interno

from autogluon.tabular import TabularPredictor

_original_kfold = kfold_validation  # guardar por si acaso

def kfold_validation_safe(
    task, dataset, setting,
    comp_model_key="chemberta_medium",
    prot_model_key="large",
    preset="good_quality",
    ex_model=[],
    num_gpus=1,
    # features reducidos opcionales
    comp_feat_override=None,
    prot_feat_override=None,
):
    """
    Wrapper de kfold_validation con:
    - num_bag_folds=0  → deshabilita bagging interno (principal causa de OOM)
    - num_stack_levels=0 → sin stacking (menos workers Ray)
    - Soporte para features pre-reducidos por PCA
    """
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy import stats

    print(f"\n🚀 DTIAM V5-1.3 [SAFE MODE]: {task.upper()} | {dataset} | {setting}")
    comp_params = PLM_CATALOG["compound"][comp_model_key]["params"]
    prot_params = PLM_CATALOG["protein"][prot_model_key]["params"]
    print(f"   PLMs: ChemBERTa-{comp_params} + ESM-2-{prot_params}")
    print(f"   Preset: {preset} | num_bag_folds=0 | num_stack_levels=0")

    task_cfg = {
        "dti": {"base": PROJECT_ROOT/"data"/"dti"/dataset, "k": 10,
                "metric": "roc_auc", "cols": ["AUROC","AUPR"]},
        "dta": {"base": PROJECT_ROOT/"data"/"dta"/dataset, "k": 5,
                "metric": "root_mean_squared_error",
                "cols": ["RMSE","MSE","Pearson","Spearman","CI"]},
        "moa": {"base": PROJECT_ROOT/"data"/"moa"/dataset, "k": 5,
                "metric": "roc_auc", "cols": ["AUROC","AUPR"]},
    }[task]

    # Cargar features — usar override (PCA) si está disponible
    if comp_feat_override is not None:
        comp_feat = comp_feat_override
        print(f"   💊 Features: PCA reducidos ({len(next(iter(comp_feat.values())))}d)")
    else:
        comp_feat = feature_store.load(f"comp_{comp_model_key}")

    if prot_feat_override is not None:
        prot_feat = prot_feat_override
        print(f"   🧬 Features: PCA reducidos ({len(next(iter(prot_feat.values())))}d)")
    else:
        prot_feat = feature_store.load(f"prot_esm2_{prot_model_key}")

    folds_path = task_cfg["base"] / "data_folds" / setting
    res_all    = pd.DataFrame(columns=task_cfg["cols"])
    start_time = time.time()

    for i in range(task_cfg["k"]):
        fold_start = time.time()
        print(f"\n🔄 Fold {i+1}/{task_cfg['k']}...", end=" ", flush=True)

        train_data, test_data = load_fold_data(folds_path, i, comp_feat, prot_feat)
        test_nolab = test_data.drop(columns=["y"])

        fold_model_dir = str(MODELS_PATH / f"{task}_{dataset}_{setting}_fold{i}")

        predictor = TabularPredictor(
            label        = "y",
            eval_metric  = task_cfg["metric"],
            path         = fold_model_dir,
            verbosity    = 0
        ).fit(
            train_data       = train_data,
            excluded_model_types = ex_model,
            presets          = preset,
            num_gpus         = num_gpus,
            # ── Las 3 líneas que evitan el OOM ──────────────────────────────
            num_bag_folds    = 0,   # sin bagging → sin workers Ray paralelos
            num_bag_sets     = 1,   # un solo set
            num_stack_levels = 0,   # sin stacking
        )

        if task == "dta":
            P   = np.array(predictor.predict(test_nolab))
            G   = np.array(test_data["y"])
            from scipy import stats
            ret = [
                float(np.sqrt(np.mean((G-P)**2))),
                float(np.mean((G-P)**2)),
                float(stats.pearsonr(G,P)[0]),
                float(stats.spearmanr(G,P)[0]),
                float(sum((p>q)==(g>h) for i,(g,p) in enumerate(zip(G,P))
                          for h,q in list(zip(G,P))[i+1:]
                          if g!=h) /
                     max(1, sum(1 for i,g in enumerate(G)
                                for h in list(G)[i+1:] if g!=h)))
            ]
            print(f"RMSE={ret[0]:.4f} | CI={ret[4]:.4f}")
            res_all.loc[i] = ret
        else:
            probs  = predictor.predict_proba(test_nolab)
            probs1 = probs.iloc[:,1] if isinstance(probs, pd.DataFrame) else probs
            y_true = np.array(test_data["y"])
            auroc  = roc_auc_score(y_true, np.array(probs1))
            aupr   = average_precision_score(y_true, np.array(probs1))
            print(f"AUROC={auroc:.4f} | AUPR={aupr:.4f}  "
                  f"[{time.time()-fold_start:.0f}s]")
            res_all.loc[i] = [auroc, aupr]

        # Liberar RAM entre folds
        del train_data, test_data, test_nolab
        gc.collect()

    print("\n" + "="*55)
    print("📊 RESULTADOS (mean ± std)")
    print("="*55)
    summary = pd.DataFrame({"Mean": res_all.mean(), "Std": res_all.std()})
    print(summary.to_string())
    elapsed = (time.time()-start_time)/60
    print(f"\n⏱️ Total: {elapsed:.1f} min")

    # Guardar
    out_csv = RESULTS_PATH / f"{task}_{dataset}_{setting}_{comp_model_key}_{prot_model_key}_safe.csv"
    res_all.to_csv(out_csv, sep="\t")

    bundle = {
        "version": "5.1.3-safe", "task": task, "dataset": dataset,
        "setting": setting, "comp_model_key": comp_model_key,
        "prot_model_key": prot_model_key,
        "preset": preset, "num_bag_folds": 0, "num_stack_levels": 0,
        "pca_comp_dims": len(next(iter(comp_feat.values()))),
        "pca_prot_dims": len(next(iter(prot_feat.values()))),
        "n_folds": task_cfg["k"],
        "fold_dirs": [str(MODELS_PATH/f"{task}_{dataset}_{setting}_fold{i}")
                      for i in range(task_cfg["k"])],
        "best_fold": int(res_all["AUROC"].idxmax()
                         if "AUROC" in res_all else res_all["RMSE"].idxmin()),
        "metrics_mean": summary["Mean"].to_dict(),
        "metrics_std":  summary["Std"].to_dict(),
        "elapsed_min": round(elapsed, 1),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    bundle_path = RESULTS_PATH / f"bundle_{task}_{dataset}_{setting}.json"
    bundle_path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")
    print(f"💾 {out_csv.name}")
    print(f"📦 {bundle_path.name}")
    return res_all


print("✅ kfold_validation_safe() lista")

✅ kfold_validation_safe() lista


In [14]:
# ── PASO 4: Correr ────────────────────────────────────────────────────────────
results = kfold_validation_safe(
    task             = "dti",
    dataset          = "yamanishi_08",
    setting          = "warm_start",
    preset           = "good_quality",   # rápido para verificar que funciona
    comp_feat_override = comp_feat_red,  # PCA 256d
    prot_feat_override = prot_feat_red,  # PCA 256d
)


🚀 DTIAM V5-1.3 [SAFE MODE]: DTI | yamanishi_08 | warm_start
   PLMs: ChemBERTa-77M + ESM-2-650M
   Preset: good_quality | num_bag_folds=0 | num_stack_levels=0
   💊 Features: PCA reducidos (256d)
   🧬 Features: PCA reducidos (256d)

🔄 Fold 1/10... AUROC=0.9662 | AUPR=0.9708  [318s]

🔄 Fold 2/10... AUROC=0.9704 | AUPR=0.9705  [303s]

🔄 Fold 3/10... AUROC=0.9710 | AUPR=0.9733  [357s]

🔄 Fold 4/10... AUROC=0.9738 | AUPR=0.9779  [282s]

🔄 Fold 5/10... AUROC=0.9764 | AUPR=0.9762  [474s]

🔄 Fold 6/10... AUROC=0.9675 | AUPR=0.9714  [315s]

🔄 Fold 7/10... AUROC=0.9751 | AUPR=0.9806  [352s]

🔄 Fold 8/10... AUROC=0.9723 | AUPR=0.9715  [429s]

🔄 Fold 9/10... AUROC=0.9654 | AUPR=0.9708  [274s]

🔄 Fold 10/10... AUROC=0.9637 | AUPR=0.9697  [312s]

📊 RESULTADOS (mean ± std)
           Mean       Std
AUROC  0.970180  0.004343
AUPR   0.973255  0.003699

⏱️ Total: 56.9 min
💾 dti_yamanishi_08_warm_start_chemberta_medium_large_safe.csv
📦 bundle_dti_yamanishi_08_warm_start.json


In [15]:
# ── PASO 4: Correr ────────────────────────────────────────────────────────────
results = kfold_validation_safe(
    task             = "dti",
    dataset          = "yamanishi_08",
    setting          = "drug_coldstart",
    preset           = "good_quality",   # rápido para verificar que funciona
    comp_feat_override = comp_feat_red,  # PCA 256d
    prot_feat_override = prot_feat_red,  # PCA 256d
)


🚀 DTIAM V5-1.3 [SAFE MODE]: DTI | yamanishi_08 | drug_coldstart
   PLMs: ChemBERTa-77M + ESM-2-650M
   Preset: good_quality | num_bag_folds=0 | num_stack_levels=0
   💊 Features: PCA reducidos (256d)
   🧬 Features: PCA reducidos (256d)

🔄 Fold 1/10... AUROC=0.8599 | AUPR=0.8779  [402s]

🔄 Fold 2/10... AUROC=0.8352 | AUPR=0.8275  [233s]

🔄 Fold 3/10... AUROC=0.7684 | AUPR=0.8104  [355s]

🔄 Fold 4/10... AUROC=0.7400 | AUPR=0.7410  [282s]

🔄 Fold 5/10... AUROC=0.8438 | AUPR=0.8358  [295s]

🔄 Fold 6/10... AUROC=0.8372 | AUPR=0.8516  [385s]

🔄 Fold 7/10... AUROC=0.7819 | AUPR=0.8114  [269s]

🔄 Fold 8/10... AUROC=0.8582 | AUPR=0.8742  [307s]

🔄 Fold 9/10... AUROC=0.7506 | AUPR=0.7662  [285s]

🔄 Fold 10/10... AUROC=0.6911 | AUPR=0.7369  [343s]

📊 RESULTADOS (mean ± std)
           Mean       Std
AUROC  0.796620  0.058347
AUPR   0.813304  0.050920

⏱️ Total: 52.6 min
💾 dti_yamanishi_08_drug_coldstart_chemberta_medium_large_safe.csv
📦 bundle_dti_yamanishi_08_drug_coldstart.json


In [16]:
# ── PASO 4: Correr ────────────────────────────────────────────────────────────
results = kfold_validation_safe(
    task             = "dti",
    dataset          = "yamanishi_08",
    setting          = "protein_coldstart",
    preset           = "good_quality",   # rápido para verificar que funciona
    comp_feat_override = comp_feat_red,  # PCA 256d
    prot_feat_override = prot_feat_red,  # PCA 256d
)


🚀 DTIAM V5-1.3 [SAFE MODE]: DTI | yamanishi_08 | protein_coldstart
   PLMs: ChemBERTa-77M + ESM-2-650M
   Preset: good_quality | num_bag_folds=0 | num_stack_levels=0
   💊 Features: PCA reducidos (256d)
   🧬 Features: PCA reducidos (256d)

🔄 Fold 1/10... AUROC=0.9463 | AUPR=0.9608  [322s]

🔄 Fold 2/10... AUROC=0.9756 | AUPR=0.9807  [282s]

🔄 Fold 3/10... AUROC=0.9484 | AUPR=0.9626  [267s]

🔄 Fold 4/10... AUROC=0.9602 | AUPR=0.9718  [354s]

🔄 Fold 5/10... AUROC=0.9349 | AUPR=0.9434  [415s]

🔄 Fold 6/10... AUROC=0.9443 | AUPR=0.9513  [461s]

🔄 Fold 7/10... AUROC=0.9610 | AUPR=0.9651  [331s]

🔄 Fold 8/10... AUROC=0.9244 | AUPR=0.9284  [351s]

🔄 Fold 9/10... AUROC=0.9459 | AUPR=0.9464  [265s]

🔄 Fold 10/10... AUROC=0.9080 | AUPR=0.9274  [320s]

📊 RESULTADOS (mean ± std)
           Mean       Std
AUROC  0.944907  0.019252
AUPR   0.953792  0.017688

⏱️ Total: 56.2 min
💾 dti_yamanishi_08_protein_coldstart_chemberta_medium_large_safe.csv
📦 bundle_dti_yamanishi_08_protein_coldstart.json


In [17]:
# ── PASO 4: Correr ────────────────────────────────────────────────────────────
results = kfold_validation_safe(
    task             = "dti",
    dataset          = "yamanishi_08",
    setting          = "warm_start",
    preset           = "best_quality",   # best para comparar diferencia de tiempo de ejecución
    comp_feat_override = comp_feat_red,  # PCA 256d
    prot_feat_override = prot_feat_red,  # PCA 256d
)


🚀 DTIAM V5-1.3 [SAFE MODE]: DTI | yamanishi_08 | warm_start
   PLMs: ChemBERTa-77M + ESM-2-650M
   Preset: best_quality | num_bag_folds=0 | num_stack_levels=0
   💊 Features: PCA reducidos (256d)
   🧬 Features: PCA reducidos (256d)

🔄 Fold 1/10... AUROC=0.9667 | AUPR=0.9711  [3177s]

🔄 Fold 2/10... AUROC=0.9770 | AUPR=0.9785  [3397s]

🔄 Fold 3/10... AUROC=0.9647 | AUPR=0.9619  [3561s]

🔄 Fold 4/10... AUROC=0.9752 | AUPR=0.9777  [3298s]

🔄 Fold 5/10... AUROC=0.9814 | AUPR=0.9839  [3392s]

🔄 Fold 6/10... AUROC=0.9683 | AUPR=0.9741  [2999s]

🔄 Fold 7/10... AUROC=0.9712 | AUPR=0.9772  [3229s]

🔄 Fold 8/10... AUROC=0.9711 | AUPR=0.9687  [2946s]

🔄 Fold 9/10... AUROC=0.9687 | AUPR=0.9727  [3602s]

🔄 Fold 10/10... AUROC=0.9726 | AUPR=0.9772  [3526s]

📊 RESULTADOS (mean ± std)
           Mean       Std
AUROC  0.971698  0.005067
AUPR   0.974309  0.006098

⏱️ Total: 552.1 min
💾 dti_yamanishi_08_warm_start_chemberta_medium_large_safe.csv
📦 bundle_dti_yamanishi_08_warm_start.json


In [18]:
# ── PASO 4: Correr ────────────────────────────────────────────────────────────
results = kfold_validation_safe(
    task             = "dti",
    dataset          = "yamanishi_08",
    setting          = "drug_coldstart",
    preset           = "best_quality",   # best para comparar diferencia de tiempo de ejecución
    comp_feat_override = comp_feat_red,  # PCA 256d
    prot_feat_override = prot_feat_red,  # PCA 256d
)


🚀 DTIAM V5-1.3 [SAFE MODE]: DTI | yamanishi_08 | drug_coldstart
   PLMs: ChemBERTa-77M + ESM-2-650M
   Preset: best_quality | num_bag_folds=0 | num_stack_levels=0
   💊 Features: PCA reducidos (256d)
   🧬 Features: PCA reducidos (256d)

🔄 Fold 1/10... AUROC=0.8816 | AUPR=0.8861  [3434s]

🔄 Fold 2/10... AUROC=0.8390 | AUPR=0.8348  [3202s]

🔄 Fold 3/10... AUROC=0.8056 | AUPR=0.8303  [3325s]

🔄 Fold 4/10... AUROC=0.7372 | AUPR=0.7312  [3629s]

🔄 Fold 5/10... AUROC=0.8441 | AUPR=0.8459  [3697s]

🔄 Fold 6/10... AUROC=0.8594 | AUPR=0.8745  [3791s]

🔄 Fold 7/10... AUROC=0.7572 | AUPR=0.7959  [3643s]

🔄 Fold 8/10... AUROC=0.8644 | AUPR=0.8820  [3457s]

🔄 Fold 9/10... AUROC=0.7930 | AUPR=0.7867  [3394s]

🔄 Fold 10/10... AUROC=0.7122 | AUPR=0.7488  [3453s]

📊 RESULTADOS (mean ± std)
           Mean       Std
AUROC  0.809369  0.058206
AUPR   0.821599  0.054559

⏱️ Total: 583.8 min
💾 dti_yamanishi_08_drug_coldstart_chemberta_medium_large_safe.csv
📦 bundle_dti_yamanishi_08_drug_coldstart.json


In [19]:
# ── PASO 4: Correr ────────────────────────────────────────────────────────────
results = kfold_validation_safe(
    task             = "dti",
    dataset          = "yamanishi_08",
    setting          = "protein_coldstart",
    preset           = "best_quality",   # best para comparar diferencia de tiempo de ejecución
    comp_feat_override = comp_feat_red,  # PCA 256d
    prot_feat_override = prot_feat_red,  # PCA 256d
)


🚀 DTIAM V5-1.3 [SAFE MODE]: DTI | yamanishi_08 | protein_coldstart
   PLMs: ChemBERTa-77M + ESM-2-650M
   Preset: best_quality | num_bag_folds=0 | num_stack_levels=0
   💊 Features: PCA reducidos (256d)
   🧬 Features: PCA reducidos (256d)

🔄 Fold 1/10... AUROC=0.9499 | AUPR=0.9635  [3602s]

🔄 Fold 2/10... AUROC=0.9757 | AUPR=0.9818  [3427s]

🔄 Fold 3/10... AUROC=0.9523 | AUPR=0.9632  [3602s]

🔄 Fold 4/10... AUROC=0.9607 | AUPR=0.9701  [3425s]

🔄 Fold 5/10... AUROC=0.9456 | AUPR=0.9524  [3647s]

🔄 Fold 6/10... AUROC=0.9405 | AUPR=0.9498  [3320s]

🔄 Fold 7/10... AUROC=0.9632 | AUPR=0.9696  [3536s]

🔄 Fold 8/10... AUROC=0.9339 | AUPR=0.9387  [3417s]

🔄 Fold 9/10... AUROC=0.9430 | AUPR=0.9449  [3434s]

🔄 Fold 10/10... AUROC=0.9198 | AUPR=0.9410  [3190s]

📊 RESULTADOS (mean ± std)
           Mean       Std
AUROC  0.948452  0.015845
AUPR   0.957491  0.014278

⏱️ Total: 576.7 min
💾 dti_yamanishi_08_protein_coldstart_chemberta_medium_large_safe.csv
📦 bundle_dti_yamanishi_08_protein_coldstart.j

In [11]:
# ── PASO 4: Correr ────────────────────────────────────────────────────────────
results = kfold_validation_safe(
    task             = "dti",
    dataset          = "hetionet",
    setting          = "drug_coldstart",
    preset           = "best_quality",   # best para comparar diferencia de tiempo de ejecución
    comp_feat_override = comp_feat_red,  # PCA 256d
    prot_feat_override = prot_feat_red,  # PCA 256d
)


🚀 DTIAM V5-1.3 [SAFE MODE]: DTI | hetionet | drug_coldstart
   PLMs: ChemBERTa-77M + ESM-2-650M
   Preset: best_quality | num_bag_folds=0 | num_stack_levels=0
   💊 Features: PCA reducidos (256d)
   🧬 Features: PCA reducidos (256d)

🔄 Fold 1/10... AUROC=0.8179 | AUPR=0.8150  [3940s]

🔄 Fold 2/10... AUROC=0.8863 | AUPR=0.8828  [3919s]

🔄 Fold 3/10... AUROC=0.7829 | AUPR=0.8139  [3610s]

🔄 Fold 4/10... AUROC=0.8592 | AUPR=0.8469  [3609s]

🔄 Fold 5/10... AUROC=0.8090 | AUPR=0.8137  [3691s]

🔄 Fold 6/10... AUROC=0.7673 | AUPR=0.7554  [4241s]

🔄 Fold 7/10... AUROC=0.8837 | AUPR=0.8887  [3609s]

🔄 Fold 8/10... AUROC=0.6690 | AUPR=0.6790  [4008s]

🔄 Fold 9/10... AUROC=0.8488 | AUPR=0.8266  [3852s]

🔄 Fold 10/10... AUROC=0.8697 | AUPR=0.8741  [3734s]

📊 RESULTADOS (mean ± std)
           Mean      Std
AUROC  0.819374  0.06697
AUPR   0.819617  0.06369

⏱️ Total: 636.9 min
💾 dti_hetionet_drug_coldstart_chemberta_medium_large_safe.csv
📦 bundle_dti_hetionet_drug_coldstart.json


---
## 🔮 CELDA 9: DTIPredictor v2 — Inferencia Standalone

In [20]:
class DTIPredictor:
    """
    Predictor standalone sin pickle en ningún punto.
    Carga el bundle JSON + HDF5 + AutoGluon.

    Uso rápido:
        pred = DTIPredictor.from_bundle('results/bundle_dti_yamanishi_08_warm_start.json')
        result = pred.predict(smiles='CC(C)Cc1ccc(cc1)...', sequence='MKTAYIAK...')
        print(result['interpretation'])
    """

    def __init__(
        self,
        bundle_path: str | None = None,
        task: str = "dti",
        dataset: str = "yamanishi_08",
        setting: str = "warm_start",
        comp_model_key: str = "chemberta_medium",
        prot_model_key: str = "large",
        fold_idx: int = 0,
    ):
        # Cargar desde bundle si se proporciona
        if bundle_path:
            bundle = json.loads(Path(bundle_path).read_text())
            task           = bundle["task"]
            dataset        = bundle["dataset"]
            setting        = bundle["setting"]
            comp_model_key = bundle["comp_model_key"]
            prot_model_key = bundle["prot_model_key"]
            fold_idx       = bundle.get("best_fold", 0)
            comp_store     = bundle["comp_store"]
            prot_store     = bundle["prot_store"]
            fold_dir       = bundle["fold_dirs"][fold_idx]
        else:
            comp_store = f"comp_{comp_model_key}"
            prot_store = f"prot_esm2_{prot_model_key}"
            fold_dir   = str(MODELS_PATH / f"{task}_{dataset}_{setting}_fold{fold_idx}")

        self.task           = task
        self.comp_model_key = comp_model_key
        self.prot_model_key = prot_model_key
        self.comp_cfg       = PLM_CATALOG["compound"][comp_model_key]
        self.prot_cfg       = PLM_CATALOG["protein"][prot_model_key]

        # ── AutoGluon ──────────────────────────────────────────────────────────
        print(f"📂 Cargando AutoGluon desde: {fold_dir}")
        self.predictor = TabularPredictor.load(fold_dir)

        # ── PLMs ───────────────────────────────────────────────────────────────
        from transformers import AutoTokenizer, AutoModel as HFModel
        print(f"🧪 Cargando ChemBERTa ({self.comp_cfg['name']})...")
        self.comp_tok = AutoTokenizer.from_pretrained(self.comp_cfg["name"])
        self.comp_mod = HFModel.from_pretrained(self.comp_cfg["name"])
        if DEVICE.type == "cuda": self.comp_mod = self.comp_mod.half()
        self.comp_mod = self.comp_mod.to(DEVICE).eval()

        print(f"🧬 Cargando ESM-2 ({self.prot_cfg['name']})...")
        self.prot_tok = AutoTokenizer.from_pretrained(self.prot_cfg["name"])
        self.prot_mod = HFModel.from_pretrained(self.prot_cfg["name"])
        if DEVICE.type == "cuda": self.prot_mod = self.prot_mod.half()
        self.prot_mod = self.prot_mod.to(DEVICE).eval()

        print("✅ DTIPredictor v2 listo.")

    @classmethod
    def from_bundle(cls, bundle_path: str) -> "DTIPredictor":
        return cls(bundle_path=bundle_path)

    # ── Embeddings ────────────────────────────────────────────────────────────
    def _embed_compound(self, smiles: str) -> np.ndarray:
        input_format = self.comp_cfg["input_format"]
        if input_format == "selfies":
            text = UnifiedRegistry.smiles_to_selfies(smiles)
            if text is None:
                raise ValueError(f"No se pudo convertir a SELFIES: {smiles[:40]}")
        else:
            text = UnifiedRegistry.canonicalize_smiles(smiles)
            if text is None:
                raise ValueError(f"SMILES inválido: {smiles[:40]}")

        enc = self.comp_tok([text], padding=True, truncation=True,
                             max_length=512, return_tensors="pt")
        with torch.no_grad():
            out = self.comp_mod(input_ids=enc["input_ids"].to(DEVICE),
                                attention_mask=enc["attention_mask"].to(DEVICE))
        return out.last_hidden_state[:, 0, :].cpu().float().numpy().reshape(-1)

    def _embed_protein(self, sequence: str) -> np.ndarray:
        seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", sequence.upper())[:1022]
        enc = self.prot_tok([seq], return_tensors="pt", padding=True,
                             truncation=True, max_length=1024)
        with torch.no_grad():
            out = self.prot_mod(input_ids=enc["input_ids"].to(DEVICE),
                                attention_mask=enc["attention_mask"].to(DEVICE))
        h    = out.last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float().to(DEVICE)
        emb  = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return emb.cpu().float().numpy().reshape(-1)

    # ── Predicción ────────────────────────────────────────────────────────────
    def predict(self, smiles: str, sequence: str) -> dict:
        """
        Predice interacción/afinidad entre un compuesto y una proteína.

        Returns:
            dict con keys: score, interpretation, (label, confidence para DTI/MOA)
        """
        cf = self._embed_compound(smiles)
        pf = self._embed_protein(sequence)

        cols = [f"c{i}" for i in range(len(cf))] + [f"p{i}" for i in range(len(pf))]
        x    = pd.DataFrame([np.concatenate([cf, pf])], columns=cols)

        if self.task == "dta":
            score = float(self.predictor.predict(x).iloc[0])
            return {
                "score": score,
                "type": "regression",
                "interpretation": f"Afinidad predicha: {score:.4f}"
            }
        else:
            probs  = self.predictor.predict_proba(x)
            prob1  = float(probs.iloc[0, 1] if isinstance(probs, pd.DataFrame) else probs.iloc[0])
            label  = int(prob1 >= 0.5)
            conf   = "Alta" if abs(prob1-0.5) > 0.3 else ("Media" if abs(prob1-0.5) > 0.15 else "Baja")
            tag    = "✅ INTERACCIÓN" if label else "❌ NO INTERACCIÓN"
            return {
                "score": prob1, "label": label,
                "confidence": conf, "type": "classification",
                "interpretation": f"{tag}  P={prob1:.4f}  Confianza: {conf}"
            }

    def predict_batch(self, pairs: list[tuple[str, str]]) -> pd.DataFrame:
        """
        Predice múltiples pares (smiles, sequence).
        Maneja errores individuales sin abortar el batch.
        """
        results = []
        for i, (smi, seq) in enumerate(tqdm(pairs, desc="🔮 Prediciendo")):
            try:
                res = self.predict(smi, seq)
                res.update({"smiles": smi[:40], "sequence": seq[:20]+"...", "error": None})
            except Exception as e:
                res = {"smiles": smi[:40], "sequence": seq[:20]+"...",
                       "score": None, "interpretation": "ERROR", "error": str(e)}
            results.append(res)
        return pd.DataFrame(results)


# ── Ejemplo de uso ─────────────────────────────────────────────────────────────
print("✅ DTIPredictor v2 definido.")
print()
print("Uso:")
print("  # Opción 1 — desde bundle (recomendado):")
print("  pred = DTIPredictor.from_bundle('results/bundle_dti_yamanishi_08_warm_start.json')")
print()
print("  # Opción 2 — manual:")
print("  pred = DTIPredictor(task='dti', dataset='yamanishi_08', setting='warm_start')")
print()
print("  # Predicción:")
print("  r = pred.predict(smiles='CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O', sequence='MKTAYIAK...')")
print("  print(r['interpretation'])")


✅ DTIPredictor v2 definido.

Uso:
  # Opción 1 — desde bundle (recomendado):
  pred = DTIPredictor.from_bundle('results/bundle_dti_yamanishi_08_warm_start.json')

  # Opción 2 — manual:
  pred = DTIPredictor(task='dti', dataset='yamanishi_08', setting='warm_start')

  # Predicción:
  r = pred.predict(smiles='CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O', sequence='MKTAYIAK...')
  print(r['interpretation'])


---
## 📊 CELDA 10: Comparativa con Literatura + Benchmarking Automático

In [21]:
# ── Tabla de referencia ──────────────────────────────────────────────────────
literature_dti = pd.DataFrame([
    {"Model":"KBMF2K",                    "Year":2012,"Drug_feat":"Fingerprint","Prot_feat":"Spectrum",   "AUROC":0.900,"AUPR":0.670,"Ref":"Gonen 2012"},
    {"Model":"DeepDTI",                   "Year":2019,"Drug_feat":"ECFP4",      "Prot_feat":"AAC",        "AUROC":0.950,"AUPR":0.800,"Ref":"Wen 2017"},
    {"Model":"DeepConv-DTI",              "Year":2019,"Drug_feat":"Morgan",     "Prot_feat":"CNN-seq",    "AUROC":0.958,"AUPR":0.825,"Ref":"Lee 2019"},
    {"Model":"MolBERT+ESM",               "Year":2021,"Drug_feat":"MolBERT",   "Prot_feat":"ESM-1b",     "AUROC":0.971,"AUPR":0.861,"Ref":"Shin 2019"},
    {"Model":"DTIAM v4 (BerMol+ESM2)",   "Year":2024,"Drug_feat":"BerMol",    "Prot_feat":"ESM-2 650M", "AUROC":0.975,"AUPR":0.870,"Ref":"Tu 2024"},
    {"Model":"DTIAM V5-1.3 (This work)", "Year":2026,"Drug_feat":"ChemBERTa-77M","Prot_feat":"ESM-2 650M","AUROC":None,"AUPR":None,"Ref":"This work"},
])

literature_dta = pd.DataFrame([
    {"Model":"KronRLS",                  "Year":2014,"Drug_feat":"Pubchem",   "Prot_feat":"SW-score",    "MSE":0.379,"CI":0.871,"Pearson":0.630,"Ref":"Pahikkala 2014"},
    {"Model":"DeepDTA",                  "Year":2018,"Drug_feat":"CNN-SMILES","Prot_feat":"CNN-seq",     "MSE":0.261,"CI":0.878,"Pearson":0.790,"Ref":"Öztürk 2018"},
    {"Model":"GraphDTA (GAT)",           "Year":2021,"Drug_feat":"GAT",       "Prot_feat":"CNN-seq",     "MSE":0.245,"CI":0.881,"Pearson":0.829,"Ref":"Nguyen 2021"},
    {"Model":"MGraphDTA",                "Year":2022,"Drug_feat":"GNN",       "Prot_feat":"CNN-seq",     "MSE":0.212,"CI":0.895,"Pearson":0.847,"Ref":"Yang 2022"},
    {"Model":"DTIAM v4 (BerMol+ESM2)", "Year":2024,"Drug_feat":"BerMol",    "Prot_feat":"ESM-2 650M",  "MSE":0.195,"CI":0.911,"Pearson":0.862,"Ref":"Tu 2024"},
    {"Model":"DTIAM V5-1.3 (This work)","Year":2026,"Drug_feat":"ChemBERTa-77M","Prot_feat":"ESM-2 650M","MSE":None,"CI":None,"Pearson":None,"Ref":"This work"},
])


def update_benchmark(results_df: pd.DataFrame, task: str) -> None:
    """
    Actualiza la tabla de literatura con los resultados obtenidos
    y guarda benchmark.json para trazabilidad.
    """
    mean = results_df.mean()

    if task in ["dti", "moa"]:
        idx = literature_dti[literature_dti["Ref"] == "This work"].index
        literature_dti.loc[idx, "AUROC"] = round(float(mean["AUROC"]), 4)
        literature_dti.loc[idx, "AUPR"]  = round(float(mean["AUPR"]),  4)
        print(f"✅ DTI benchmark actualizado: AUROC={mean['AUROC']:.4f} | AUPR={mean['AUPR']:.4f}")
    elif task == "dta":
        idx = literature_dta[literature_dta["Ref"] == "This work"].index
        literature_dta.loc[idx, "MSE"]     = round(float(mean["MSE"]),     4)
        literature_dta.loc[idx, "CI"]      = round(float(mean["CI"]),      4)
        literature_dta.loc[idx, "Pearson"] = round(float(mean["Pearson"]), 4)
        print(f"✅ DTA benchmark actualizado: MSE={mean['MSE']:.4f} | CI={mean['CI']:.4f}")

    # Guardar benchmark completo en JSON (sin pickle)
    benchmark = {
        "dti": literature_dti.to_dict(orient="records"),
        "dta": literature_dta.to_dict(orient="records"),
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    bpath = RESULTS_PATH / "benchmark.json"
    bpath.write_text(json.dumps(benchmark, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"💾 Guardado: {bpath}")


def show_benchmark(task: str = "dti"):
    """Muestra la tabla comparativa con highlight."""
    if task in ["dti","moa"]:
        return (literature_dti
                .style
                .format({"AUROC":"{:.3f}","AUPR":"{:.3f}"}, na_rep="[Run model]")
                .highlight_max(subset=["AUROC","AUPR"], color="#d4edda", axis=0)
                .set_caption("Benchmark DTI — Yamanishi '08 / Warm-start"))
    else:
        return (literature_dta
                .style
                .format({"MSE":"{:.3f}","CI":"{:.3f}","Pearson":"{:.3f}"}, na_rep="[Run model]")
                .highlight_min(subset=["MSE"], color="#d4edda", axis=0)
                .highlight_max(subset=["CI","Pearson"], color="#d4edda", axis=0)
                .set_caption("Benchmark DTA — Davis"))


print("✅ Tablas de benchmark listas.")
print("   Uso: show_benchmark('dti')  |  update_benchmark(results, 'dti')")
show_benchmark("dti")


✅ Tablas de benchmark listas.
   Uso: show_benchmark('dti')  |  update_benchmark(results, 'dti')


,Model,Year,Drug_feat,Prot_feat,AUROC,AUPR,Ref
0,KBMF2K,2012,Fingerprint,Spectrum,0.900,0.670,Gonen 2012
1,DeepDTI,2019,ECFP4,AAC,0.950,0.800,Wen 2017
2,DeepConv-DTI,2019,Morgan,CNN-seq,0.958,0.825,Lee 2019
3,MolBERT+ESM,2021,MolBERT,ESM-1b,0.971,0.861,Shin 2019
4,DTIAM v4 (BerMol+ESM2),2024,BerMol,ESM-2 650M,0.975,0.870,Tu 2024
5,DTIAM V5-1.3 (This work),2026,ChemBERTa-77M,ESM-2 650M,[Run model],[Run model],This work


---
## 🏃 CELDA 11: Ejecución Completa

In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — ajusta y ejecuta esta celda para correr el pipeline completo
# ══════════════════════════════════════════════════════════════════════════════

RUN_CONFIG = {
    # ── PLMs ─────────────────────────────────────────────────────────────────
    # Opción A (SMILES): 'chemberta_light' | 'chemberta_medium' | 'chemberta_base'
    # Opción B (SELFIES): 'selformer'
    "compound_model": "chemberta_medium",
    "protein_model":  "large",

    # ── Tarea ────────────────────────────────────────────────────────────────
    "task":    "dti",          # 'dti' | 'dta' | 'moa'
    "dataset": "yamanishi_08",
    "setting": "warm_start",   # 'warm_start' | 'drug_coldstart' | 'protein_coldstart'

    # ── AutoGluon ────────────────────────────────────────────────────────────
    "preset":   "best_quality",  # 'best_quality' | 'good_quality' | 'fast'
    "ex_model": [],
    "num_gpus": 1,

    # ── Pasos a ejecutar ─────────────────────────────────────────────────────
    "run_split":          False,  # Regenerar folds (solo si no existen)
    "run_comp_features":  False,  # Re-extraer embeddings de compuestos
    "run_prot_features":  False,  # Re-extraer embeddings de proteínas
    "run_training":       True,
    "run_benchmark":      True,
}

# ── Mostrar config ────────────────────────────────────────────────────────────
comp_fmt = PLM_CATALOG["compound"][RUN_CONFIG["compound_model"]].get("input_format","smiles")
print(f"📋 Config activa:")
print(f"   Representación: {'Opción B — SELFIES' if comp_fmt=='selfies' else 'Opción A — SMILES canónico'}")
for k, v in RUN_CONFIG.items():
    print(f"   {k:22s}: {v}")
print()

# ── Splits ────────────────────────────────────────────────────────────────────
if RUN_CONFIG["run_split"]:
    if RUN_CONFIG["task"] in ["dti","moa"]:
        split_dti(RUN_CONFIG["dataset"], task=RUN_CONFIG["task"])
    else:
        split_dta(RUN_CONFIG["dataset"])

# ── Features ──────────────────────────────────────────────────────────────────
if RUN_CONFIG["run_comp_features"]:
    feats = cal_comp_feat(master_drugs_df, model_key=RUN_CONFIG["compound_model"])
    feature_store.save(feats, f"comp_{RUN_CONFIG['compound_model']}",
                       metadata={"model_key": RUN_CONFIG["compound_model"]})

if RUN_CONFIG["run_prot_features"]:
    feats = cal_prot_feat(master_prots_df, model_key=RUN_CONFIG["protein_model"])
    feature_store.save(feats, f"prot_esm2_{RUN_CONFIG['protein_model']}",
                       metadata={"model_key": RUN_CONFIG["protein_model"]})

# ── Entrenamiento ─────────────────────────────────────────────────────────────
if RUN_CONFIG["run_training"]:
    results = kfold_validation(
        task           = RUN_CONFIG["task"],
        dataset        = RUN_CONFIG["dataset"],
        setting        = RUN_CONFIG["setting"],
        comp_model_key = RUN_CONFIG["compound_model"],
        prot_model_key = RUN_CONFIG["protein_model"],
        preset         = RUN_CONFIG["preset"],
        ex_model       = RUN_CONFIG["ex_model"],
        num_gpus       = RUN_CONFIG["num_gpus"],
    )

# ── Benchmark ─────────────────────────────────────────────────────────────────
if RUN_CONFIG["run_benchmark"] and RUN_CONFIG["run_training"]:
    update_benchmark(results, RUN_CONFIG["task"])
    display(show_benchmark(RUN_CONFIG["task"]))

print("\n🏁 Pipeline completo.")


📋 Config activa:
   Representación: Opción A — SMILES canónico
   compound_model        : chemberta_medium
   protein_model         : large
   task                  : dti
   dataset               : yamanishi_08
   setting               : warm_start
   preset                : best_quality
   ex_model              : []
   num_gpus              : 1
   run_split             : False
   run_comp_features     : False
   run_prot_features     : False
   run_training          : True
   run_benchmark         : True


🚀 DTIAM V5-1.3: DTI | yamanishi_08 | warm_start
   PLMs: ChemBERTa-77M + ESM-2-650M
📥 Cargando features: comp_chemberta_medium.h5, prot_esm2_large.h5
✅ Cargado: comp_chemberta_medium.h5  (18272 entidades, dim=384)
✅ Cargado: prot_esm2_large.h5  (18903 entidades, dim=1280)

🔄 Fold 1/10... AUROC=0.9704 | AUPR=0.9741
   ⏱️ 2722.5s

🔄 Fold 2/10... AUROC=0.9748 | AUPR=0.9730
   ⏱️ 2751.8s

🔄 Fold 3/10... AUROC=0.9747 | AUPR=0.9762
   ⏱️ 2838.1s

🔄 Fold 4/10... AUROC=0.9757 | AUPR=0.9738


,Model,Year,Drug_feat,Prot_feat,AUROC,AUPR,Ref
0,KBMF2K,2012,Fingerprint,Spectrum,0.900,0.670,Gonen 2012
1,DeepDTI,2019,ECFP4,AAC,0.950,0.800,Wen 2017
2,DeepConv-DTI,2019,Morgan,CNN-seq,0.958,0.825,Lee 2019
3,MolBERT+ESM,2021,MolBERT,ESM-1b,0.971,0.861,Shin 2019
4,DTIAM v4 (BerMol+ESM2),2024,BerMol,ESM-2 650M,0.975,0.870,Tu 2024
5,DTIAM V5-1.3 (This work),2026,ChemBERTa-77M,ESM-2 650M,0.975,0.976,This work



🏁 Pipeline completo.


# Viene la parte del screening de un solo compuesto. 
## La etapa 1 es la validación y embedding del compuesto de entrada.


In [12]:
def screen_compound(
    smiles: str,
    top_k: int = 50,
    protein_fasta_path: str | None = None,
    model_bundle_path: str | None = None,
    expand_kegg: bool = True,
    fold_confidence: bool = True,
) -> pd.DataFrame:
    """
    Dado un SMILES, encuentra las proteínas con mayor probabilidad
    de interacción ordenadas por score.

    Args:
        smiles:             SMILES del compuesto a evaluar
        top_k:              cuántas proteínas retornar (default 50)
        protein_fasta_path: si None → usa el HDF5 del registry
                            si path → genera embeddings en el momento
        model_bundle_path:  si None → usa el mejor bundle disponible
                            si path → usa ese bundle específico
        expand_kegg:        si True → agrega pathways de KEGG
        fold_confidence:    si True → calcula confianza entre los 10 folds

    Returns:
        DataFrame con ranking, scores, confianza y metadata KEGG
    """

# La funcion en si, dividida en cinco etapas. 


In [ ]:
def screen_compound(
    smiles: str,
    top_k: int = 50,
    protein_fasta_path: str | None = None,
    model_bundle_path: str | None = None,
    expand_kegg: bool = True,
    fold_confidence: bool = True,
) -> pd.DataFrame:
    """
    Dado un SMILES, encuentra las proteínas con mayor probabilidad
    de interacción ordenadas por score.

    Args:
        smiles:             SMILES del compuesto a evaluar
        top_k:              cuántas proteínas retornar (default 50)
        protein_fasta_path: si None → usa el HDF5 del registry
                            si path → genera embeddings en el momento
        model_bundle_path:  si None → usa el mejor bundle disponible
                            si path → usa ese bundle específico
        expand_kegg:        si True → agrega pathways de KEGG
        fold_confidence:    si True → calcula confianza entre los 10 folds

    Returns:
        DataFrame con ranking, scores, confianza y metadata KEGG
    """
import requests
import time
from pathlib import Path

# ══════════════════════════════════════════════════════════════════
# ETAPA 1 — Validar y canonicalizar el SMILES
# ══════════════════════════════════════════════════════════════════
print(f"🔍 Validando SMILES...")
# Canonicalizar usa los 3 intentos de tolerancia que ya tienes
canon_smiles = UnifiedRegistry.canonicalize_smiles(smiles)
if canon_smiles is None:
    raise ValueError(
    f"❌ SMILES inválido: '{smiles}'\n"
    f"   Verifica la estructura en: https://pubchem.ncbi.nlm.nih.gov//edit3/index.html"
    )

# Avisar si el SMILES cambió al canonicalizar
if canon_smiles != smiles:
    print(f"   ℹ️  SMILES canonicalizado: {smiles[:40]}... → {canon_smiles[:40]}...")
else:
    print(f"   ✅ SMILES válido: {canon_smiles[:60]}...")
# ══════════════════════════════════════════════════════════════════
# ETAPA 2 — Generar embedding del compuesto (UNA SOLA VEZ)
# ══════════════════════════════════════════════════════════════════
print(f"🧪 Generando embedding del compuesto...")
t0 = time.time()

from transformers import AutoTokenizer, AutoModel as HFModel

# Detectar qué modelo de compuestos usar según el bundle
# Por ahora usamos el default del PLM_CATALOG
comp_cfg = PLM_CATALOG["compound"]["chemberta_medium"]

comp_tok = AutoTokenizer.from_pretrained(comp_cfg["name"])
comp_mod = HFModel.from_pretrained(comp_cfg["name"])
if DEVICE.type == "cuda":
    comp_mod = comp_mod.half()
comp_mod = comp_mod.to(DEVICE).eval()

# Tokenizar y generar embedding — igual que en DTIPredictor
enc = comp_tok(
    [canon_smiles],
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
with torch.no_grad():
    out = comp_mod(
        input_ids=enc["input_ids"].to(DEVICE),
        attention_mask=enc["attention_mask"].to(DEVICE)
    )

# Vector 1D de 768 números — el "ADN numérico" del compuesto
comp_vec = out.last_hidden_state[:, 0, :].cpu().float().numpy().reshape(-1)

# Liberar GPU inmediatamente — no la necesitamos más en esta etapa
comp_mod.cpu()
del comp_mod, comp_tok
torch.cuda.empty_cache()

print(f"   ✅ Embedding generado: {comp_vec.shape[0]}d en {time.time()-t0:.1f}s")

# ══════════════════════════════════════════════════════════════════
# ETAPA 3 — Cargar librería de proteínas
# ══════════════════════════════════════════════════════════════════
print(f"🧬 Cargando librería de proteínas...")
t0 = time.time()

if protein_fasta_path is None:
    # Opción A (default) — cargar desde HDF5
    prot_feat = feature_store.load("prot_esm2_large")
    prot_uids = list(prot_feat.keys())
    prot_vecs = np.stack([prot_feat[uid] for uid in prot_uids])
    print(f"   ✅ {len(prot_uids)} proteínas cargadas desde HDF5 en {time.time()-t0:.1f}s")

else:
    # Opción B — generar embeddings desde FASTA
    print(f"   📂 Generando embeddings desde: {protein_fasta_path}")
    fasta_df = _parse_fasta(protein_fasta_path)  # función auxiliar (abajo)
    prot_feat_new = cal_prot_feat(fasta_df, model_key="large")
    prot_uids = list(prot_feat_new.keys())
    prot_vecs  = np.stack([prot_feat_new[uid] for uid in prot_uids])
    print(f"   ✅ {len(prot_uids)} proteínas procesadas en {time.time()-t0:.1f}s")
# ══════════════════════════════════════════════════════════════════
# ETAPA 4 — Scoring masivo (batch, no loop)
# ══════════════════════════════════════════════════════════════════
print(f"🔮 Calculando scores para {len(prot_uids)} proteínas...")
t0 = time.time()

# Detectar el mejor bundle disponible
bundle_path = _get_best_bundle(model_bundle_path)
bundle      = json.loads(Path(bundle_path).read_text())

n_prot  = len(prot_uids)
n_comp  = len(comp_vec)
n_prot_ = prot_vecs.shape[1]

    # Repetir el vector del compuesto n_prot veces
    # comp_vec es (768,) → comp_matrix es (n_prot × 768)
comp_matrix = np.tile(comp_vec, (n_prot, 1))

# Concatenar → X es (n_prot × 2048)
X = np.concatenate([comp_matrix, prot_vecs], axis=1)

    # Nombres de columnas que AutoGluon espera
cols = (
    [f"c{i}" for i in range(n_comp)] +
    [f"p{i}" for i in range(n_prot_)]
)
X_df = pd.DataFrame(X, columns=cols)

# ── Scoring por fold para confianza ──────────────────────────────
all_fold_scores = []

fold_dirs = bundle["fold_dirs"]
for fold_dir in fold_dirs:
    if not Path(fold_dir).exists():
        continue
    predictor = TabularPredictor.load(fold_dir, require_version_match=False)
    probs     = predictor.predict_proba(X_df)
    scores_1  = probs.iloc[:, 1].values  # probabilidad clase 1
    all_fold_scores.append(scores_1)

    # all_fold_scores es lista de n_folds arrays de longitud n_prot
fold_matrix   = np.stack(all_fold_scores)         # (n_folds × n_prot)
scores_mean   = fold_matrix.mean(axis=0)           # (n_prot,)
scores_std    = fold_matrix.std(axis=0)            # (n_prot,)
# Cuántos folds predicen interacción positiva
folds_pos     = (fold_matrix >= 0.5).sum(axis=0)  # (n_prot,)

print(f"   ✅ {n_prot} scores calculados en {time.time()-t0:.1f}s")

# ══════════════════════════════════════════════════════════════════
# ETAPA 5 — Construir resultado con metadata y KEGG
# ══════════════════════════════════════════════════════════════════
print(f"📊 Construyendo resultado...")

# Ordenar por score descendente y tomar top_k
top_idx   = np.argsort(scores_mean)[::-1][:top_k]

rows = []
for rank, idx in enumerate(top_idx, start=1):
    uid       = prot_uids[idx]
    score     = float(scores_mean[idx])
    std       = float(scores_std[idx])
    n_pos     = int(folds_pos[idx])
    n_folds   = len(all_fold_scores)

    # Confianza categórica
    if n_pos >= 8:
        confidence = "Alta"
    elif n_pos >= 6:
        confidence = "Media"
    else:
        confidence = "Baja"

    # Recuperar ID original del registry
    aliases    = registry.prot_aliases.get(uid, {})
    pid_orig   = next(iter(aliases.keys()), uid)
    datasets   = list(set(aliases.values()))

    row = {
        "rank":            rank,
        "prot_uid":        uid,
        "pid_original":    pid_orig,
        "score":           round(score, 4),
        "score_std":       round(std, 4),
        "confidence":      confidence,
        "folds_positive":  f"{n_pos}/{n_folds}",
        "datasets":        ", ".join(datasets),
        "smiles_input":    canon_smiles,
        "model_bundle":    Path(bundle_path).name,
    }

        # Expandir con KEGG si está habilitado
    if expand_kegg:
        kegg_info = _query_kegg(pid_orig)
        row["kegg_pathways"] = kegg_info.get("pathways", [])
        row["kegg_diseases"]  = kegg_info.get("diseases", [])
        row["kegg_url"]       = kegg_info.get("url", "")

    rows.append(row)

results_df = pd.DataFrame(rows)

print(f"\n{'═'*55}")
print(f"✅ Screening completo")
print(f"   Compuesto: {canon_smiles[:50]}...")
print(f"   Top {top_k} proteínas rankeadas")
print(f"   Top 3 hits:")
for _, r in results_df.head(3).iterrows():
    print(f"     #{r['rank']:2d} {r['pid_original']:15s} "
            f"score={r['score']:.4f} conf={r['confidence']}")
print(f"{'═'*55}")

return results_df

def _get_best_bundle(override_path: str | None = None) -> str:
    """
    Retorna el path al bundle con mayor AUROC disponible.
    Si override_path se especifica, lo usa directamente.
    """
    if override_path:
        return override_path

    bundles = list(RESULTS_PATH.glob("bundle_dti_*.json"))
    if not bundles:
        raise FileNotFoundError(
            "❌ No se encontró ningún bundle entrenado.\n"
            "   Ejecuta kfold_validation() primero."
        )

    # Leer el AUROC de cada bundle y retornar el mejor
    best_path, best_auroc = None, 0.0
    for b in bundles:
        try:
            data  = json.loads(b.read_text())
            auroc = data.get("metrics_mean", {}).get("AUROC", 0.0)
            if auroc > best_auroc:
                best_auroc = auroc
                best_path  = str(b)
        except Exception:
            continue

    print(f"   📦 Bundle seleccionado: {Path(best_path).name} (AUROC={best_auroc:.4f})")
    return best_path


def _parse_fasta(fasta_path: str) -> pd.DataFrame:
    """
    Lee un archivo FASTA y retorna DataFrame [pid, seq].
    Soporta formato estándar y multi-secuencia.
    """
    records = []
    current_id, current_seq = None, []

    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current_id:
                    records.append({
                        "pid": current_id,
                        "seq": "".join(current_seq)
                    })
                # Tomar solo el primer token del header como ID
                current_id  = line[1:].split()[0]
                current_seq = []
            elif line:
                current_seq.append(line)

    # No olvidar la última secuencia
    if current_id:
        records.append({"pid": current_id, "seq": "".join(current_seq)})

    df = pd.DataFrame(records)
    print(f"   📄 FASTA parseado: {len(df)} secuencias")
    return df


def _query_kegg(pid_original: str) -> dict:
    """
    Consulta la API REST de KEGG para obtener pathways y enfermedades.

    KEGG usa IDs tipo 'hsa:1136' (human gene 1136).
    Si el pid no tiene ese formato, intenta inferirlo.
    """
    import requests

    # Normalizar ID al formato KEGG
    if ":" in pid_original:
        kegg_id = pid_original          # ya tiene formato hsa:1136
    elif pid_original.isdigit():
        kegg_id = f"hsa:{pid_original}" # solo número → asumir humano
    else:
        return {}                       # formato desconocido, skip silencioso

    url = f"https://rest.kegg.jp/get/{kegg_id}"

    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return {}

        pathways, diseases = [], []
        for line in resp.text.splitlines():
            if line.startswith("PATHWAY"):
                # Formato: "PATHWAY     hsa04010  MAPK signaling pathway"
                parts = line.split()
                if len(parts) >= 3:
                    pathway_id   = parts[1]
                    pathway_name = " ".join(parts[2:])
                    pathways.append(f"{pathway_id}: {pathway_name}")
            elif line.startswith("DISEASE"):
                parts = line.split()
                if len(parts) >= 3:
                    disease_id   = parts[1]
                    disease_name = " ".join(parts[2:])
                    diseases.append(f"{disease_id}: {disease_name}")
            elif line.startswith("///"):
                break   # fin del registro KEGG

        return {
            "pathways": pathways,
            "diseases": diseases,
            "url":      f"https://www.kegg.jp/entry/{kegg_id}"
        }

    except requests.Timeout:
        print(f"   ⚠️  KEGG timeout para {kegg_id} — continuando sin pathways")
        return {}
    except Exception:
        return {}
    
    # Screening del Ibuprofen contra todas las proteínas del HDF5
results = screen_compound(
    smiles   = "CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O",
    top_k    = 50,
    expand_kegg = True,
)

# Ver resultado
display(results[["rank","pid_original","score","confidence",
                 "folds_positive","kegg_pathways"]].head(10))

# Exportar para Cytoscape
results.to_csv(RESULTS_PATH / "screening_ibuprofen.csv", index=False)
    


🔍 Validando SMILES...


NameError: name 'smiles' is not defined